# A robustez v2, contada do começo ao fim

Este notebook acompanha a robustez refeita com o protocolo de degradação v2 (`codigo/robustez-2026-11/`).
Ele serve para eu explorar cada resultado e, ao mesmo tempo, contar a história em ordem, com os gráficos que vão para a conversa com a orientação e para o artigo.

Duas regras valem do início ao fim:

- **nenhum número escrito à mão**: toda frase que cita um valor é gerada por uma célula de código, a partir das tabelas do pipeline;
- **nada de análise refeita aqui**: as contas estatísticas vêm de `analise/estatistica-2026-08/tabelas_v2/` (gerado por `run_all.py --v2`). O notebook lê, junta e desenha; as únicas contas novas são descritivas (medianas de caminho) e o IC do ganho estático, feito com a mesma função `bootstrap_bca_geom` do pipeline.

Rode de cima a baixo com o kernel do `.venv` de `analise/estatistica-2026-08/`.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd, duckdb
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import Markdown, display, Image

INICIO = time.time()

def acha_raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "maps").is_dir() and (q / "codigo" / "robustez-2026-11").is_dir():
            return q
    raise SystemExit("abra o notebook de dentro do repositorio")

RAIZ = acha_raiz()
ANALISE = RAIZ / "analise" / "estatistica-2026-08"
T1, T2, T2F = ANALISE / "tabelas", ANALISE / "tabelas_v2", ANALISE / "tabelas_v2_fora_treino"
MEDICOES = ANALISE / "medicoes"
V2 = Path(os.environ.get("HEURISTICAS_DADOS_V2", RAIZ / "dados" / "brutos" / "v2"))
CACHE = V2 / "cache"
FIG = RAIZ / "analise" / "notebook-2026-10" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# dados brutos de marco: HEURISTICAS_DADOS, depois dados/brutos, depois ~/Downloads
def acha_marco():
    for c in (os.environ.get("HEURISTICAS_DADOS"), RAIZ / "dados" / "brutos", Path.home() / "Downloads"):
        if c and (Path(c) / "resultados_robustez.csv").exists():
            return Path(c)
    raise SystemExit("defina HEURISTICAS_DADOS com a pasta dos CSVs de marco (ver dados/MANIFESTO.md)")

MARCO = acha_marco()
sys.path.insert(0, str(ANALISE / "scripts"))
from comum import media_geometrica, bootstrap_bca_geom, categoria, SEED
print("repositorio:", RAIZ)
print("dados v2:   ", V2)
print("dados marco:", MARCO)

In [ ]:
ESTRATEGIAS = ["Manhattan", "Fórmula", "Memória-10", "Memória-20", "Memória-50", "Memória-100"]
MEMORIAS = ESTRATEGIAS[2:]
PADRAO_PT = {"Linear": "Linear", "Organic": "Orgânico", "Radial": "Radial", "Sparse": "Esparso",
             "Stochastic": "Estocástico"}
REGIME_PT = {"com_atualizacao": "com atualização", "sem_atualizacao": "sem atualização"}
GRUPOS = ["Arena", "Área aberta", "Dungeon", "Natureza"]

# paleta Okabe-Ito para destaque, uma familia de azuis para a memoria, cinza para a Manhattan
COR = {"Manhattan": "#7a7a7a", "Fórmula": "#D55E00", "Memória-10": "#8fbfe0", "Memória-20": "#4f9bd0",
       "Memória-50": "#1f6fb2", "Memória-100": "#08306b"}
MARCA = {"Manhattan": "s", "Fórmula": "D", **{m: "o" for m in MEMORIAS}}

def estilo(e, regime="com_atualizacao"):
    sem = e in MEMORIAS and regime == "sem_atualizacao"
    return dict(color=COR[e], marker=MARCA[e], ls="--" if sem else "-", alpha=0.75 if sem else 1.0,
                markerfacecolor="white" if sem else COR[e], markeredgecolor=COR[e])

def rotulo(e, regime="com_atualizacao"):
    if e in MEMORIAS:
        return f"{e} {REGIME_PT[regime]}"
    return e

def num(x, casas=2):
    return f"{x:.{casas}f}".replace(".", ",")

def milhar(n):
    return f"{int(round(n)):,}".replace(",", ".")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11.5,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": False, "legend.frameon": False,
    "pdf.fonttype": 42, "svg.hashsalt": "historia_v2",
})

def salvar(fig, nome):
    fig.savefig(FIG / f"{nome}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    fig.savefig(FIG / f"{nome}.pdf", bbox_inches="tight", metadata={"CreationDate": None})
    print(f"exportada: figuras/{nome}.png e .pdf")

def espalha(ys, gap):
    """Posicoes de rotulo com distancia minima gap, preservando a ordem."""
    ys = np.asarray(ys, float)
    ordem = np.argsort(ys)
    pos = ys[ordem].copy()
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + gap)
    out = np.empty_like(pos)
    out[ordem] = pos
    return out

In [ ]:
le = lambda pasta, nome: pd.read_csv(pasta / f"b1_{nome}.csv", dtype={"nivel": str})
G, GF = le(T2, "global"), le(T2F, "global")
PM, PMF = le(T2, "por_mapa"), le(T2F, "por_mapa")
POSTOS, FRIEDMAN, CLIFF = le(T2, "postos"), le(T2, "friedman"), le(T2, "cliff")
PN = le(T2, "padrao_nivel")
INTER = pd.read_csv(T2 / "b1_interacao.csv")
SOB = pd.read_csv(T2 / "b1_sobrevivencia_v1_v2.csv")
REJ = pd.read_csv(T2 / "b1_rejeicao.csv")
PIV = pd.read_csv(T2 / "b1_pivos_bloqueados.csv")
COLAPSO = pd.read_csv(T2 / "b1_colapso_pivo.csv")
ANOVA_V2 = {(m, r): pd.read_csv(T2 / f"b1_anova_{m}_{r}.csv", index_col=0)
            for m in ("delta", "absoluta") for r in REGIME_PT}
ANOVA_V1 = pd.read_csv(T1 / "a1_anova_completa.csv", index_col=0)
POSTOS_V1 = pd.read_csv(T1 / "a2_postos.csv", index_col=0)
DELTA_V1 = pd.read_csv(T1 / "a2_escore_degradado.csv", index_col=0)
RESUMO = pd.read_csv(MEDICOES / "degradacao_v2_resumo.csv")

# cache Parquet dos dados brutos v2 (sem as colunas de tempo), gerado uma vez fora do Git
PQ = CACHE / "robustez_v2.parquet"
if not PQ.exists():
    t = time.time()
    duckdb.sql(f"""COPY (SELECT COLUMNS(lambda c: NOT starts_with(c, 'tempo_'))
                   FROM read_csv('{V2 / "resultados_robustez_v2.csv"}', header = true))
                   TO '{PQ}' (FORMAT parquet, COMPRESSION zstd)""")
    print(f"cache criado em {time.time() - t:.0f} s: {PQ}")
v2 = lambda sql: duckdb.sql(sql.replace("v2", f"read_parquet('{PQ}')")).df()

# grids do den501d s42 a 30% para os exemplos visuais
GRIDS = CACHE / "grids"
if not all((GRIDS / f"den501d_42_{p}_v2_30.txt").exists() for p in PADRAO_PT):
    subprocess.run(["make", "-C", str(RAIZ / "codigo" / "robustez-2026-11"), "grids", f"SAIDA={V2}"],
                   check=True, capture_output=True)

def grid(nome):
    return np.array([list(l) for l in (GRIDS / f"{nome}.txt").read_text().split("\n") if l])

def mapa(nome):
    L = (RAIZ / "maps" / f"{nome}.map").read_text().split("\n")
    h, w = int(L[1].split()[1]), int(L[2].split()[1])
    return np.array([[c == "." for c in L[4 + i][:w]] for i in range(h)])

N_MAPAS = PM[(PM.nivel == "estatico")].mapa.nunique()
N_MAPAS_F = PMF[(PMF.nivel == "estatico")].mapa.nunique()
print("mapas:", N_MAPAS, "| fora do treino:", N_MAPAS_F)

## Ato 0 — A pergunta

Quero saber **qual heurística aguenta melhor um mapa que muda**. Comparo três heurísticas do A\* em mapas de Dragon Age: Origins que vão sendo bloqueados:

- **Manhattan**: não sabe nada do mapa, só a distância em linha reta (em grade);
- **Fórmula**: uma expressão sintetizada por algoritmo genético para cada mapa, que também não guarda nada do mapa;
- **Memória**: guarda as distâncias de alguns pontos fixos (pivôs, com 10, 20, 50 ou 100 deles) até todas as células.

A memória sob degradação tem dois regimes, e eles são o centro da história:

- **com atualização**: as distâncias dos pivôs são recalculadas no mapa degradado;
- **sem atualização**: a memória continua usando as distâncias do mapa original.

In [ ]:
gl = G[(G.metrica == "absoluta") & (G.nivel == "todos")].set_index(["regime", "estrategia"])
ganho = lambda r, e: 1 / gl.loc[(r, e), "media_geometrica"]
pmd = PM[(PM.metrica == "absoluta") & (PM.nivel == "todos")]
sem = pmd[pmd.regime == "sem_atualizacao"].pivot(index="mapa", columns="estrategia", values="valor")
com = pmd[pmd.regime == "com_atualizacao"].pivot(index="mapa", columns="estrategia", values="valor")
venc_sem = int((sem.idxmin(axis=1) == "Fórmula").sum())
venc_com = int((com.idxmin(axis=1) == "Memória-100").sum())
display(Markdown(
    f"**O resultado em uma frase:** num mapa que muda, quem vence depende de a heurística poder ser atualizada. "
    f"Com atualização, a Memória-100 faz {num(ganho('com_atualizacao', 'Memória-100'))}× menos expansões que a "
    f"Manhattan e é a melhor em {venc_com} de {N_MAPAS} mapas. Sem atualização, ela cai para "
    f"{num(ganho('sem_atualizacao', 'Memória-100'))}×, e a fórmula, com {num(ganho('sem_atualizacao', 'Fórmula'))}×, "
    f"passa a ser a melhor em {venc_sem} de {N_MAPAS} mapas."))

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 2.2))
itens = [("Manhattan", "com_atualizacao", "não sabe nada do mapa"),
         ("Fórmula", "com_atualizacao", "sintetizada por AG, sem memória"),
         ("Memória-10", "com_atualizacao", "memória, distâncias recalculadas"),
         ("Memória-100", "com_atualizacao", ""),
         ("Memória-10", "sem_atualizacao", "memória, distâncias do original"),
         ("Memória-100", "sem_atualizacao", "")]
for i, (e, r, txt) in enumerate(itens):
    col, lin = i // 2, i % 2
    x0, y = col * 2.45, 1 - lin * 0.55
    ax.plot([x0, x0 + 0.5], [y, y], lw=2.2, ms=6, **estilo(e, r))
    ax.text(x0 + 0.62, y, rotulo(e, r), va="center", fontsize=9)
    if txt:
        ax.text(x0, 1.45, txt, fontsize=8.5, color="0.35")
ax.text(4.9 + 0.62, 0.15, "10, 20, 50 e 100 pivôs: do claro ao escuro", fontsize=8.5, color="0.35")
ax.set_xlim(-0.1, 7.6); ax.set_ylim(0, 1.7); ax.axis("off")
ax.set_title("Como ler os gráficos: cada heurística tem cor e marcador fixos; tracejado = memória sem atualização")
plt.show()

## Ato 1 — Os dados

**Pergunta:** em que mapas e com que problemas eu testo?

Um **par** é um problema de busca: uma origem e um destino tirados do cenário oficial do Moving AI para aquele mapa. Para cada par, o A\* roda uma vez com cada heurística, e eu conto as **expansões**: quantos nós a busca tirou da fila antes de achar o caminho. Menos expansões = heurística que guia melhor. É a métrica principal porque é determinística (o tempo de execução não é confiável aqui, ver P3).

In [ ]:
pares = RESUMO[(RESUMO.padrao == "Linear") & (RESUMO.nivel == 0.1)].groupby("mapa").pares_validos.first()
mapas = sorted(pares.index, key=lambda m: (GRUPOS.index(categoria(m)), m))
por_grupo = {g: [m for m in mapas if categoria(m) == g] for g in GRUPOS}
ncol = max(len(v) for v in por_grupo.values())
fig, axes = plt.subplots(len(GRUPOS), ncol, figsize=(1.55 * ncol, 1.9 * len(GRUPOS)))
for i, g in enumerate(GRUPOS):
    for j in range(ncol):
        ax = axes[i, j]; ax.axis("off")
        if j < len(por_grupo[g]):
            m = por_grupo[g][j]
            ax.imshow(mapa(m), cmap=ListedColormap(["#2b2b2b", "#f4f1ea"]), interpolation="nearest")
            ax.set_title(f"{m}\n{milhar(pares[m])} pares", fontsize=8, fontweight="normal",
                         loc="center")
    axes[i, 0].text(-0.15, 0.5, g, transform=axes[i, 0].transAxes, rotation=90, ha="right", va="center",
                    fontsize=10, fontweight="bold")
fig.suptitle(f"Os {len(mapas)} mapas, por perfil (escuro = obstáculo)", x=0.02, ha="left",
             fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "mapas_por_perfil")
plt.show()

In [ ]:
sementes = RESUMO.semente.nunique(); padroes = RESUMO.padrao.nunique(); niveis = RESUMO.nivel.nunique()
n_pares = int(pares.sum())
linhas = int(v2("SELECT count(*) AS n FROM v2").n[0])
display(Markdown(
    f"São {len(mapas)} mapas, com {milhar(n_pares)} pares conectados por semente ({milhar(n_pares * sementes)} no "
    f"total, somando as {sementes} sementes). Cada par é degradado por {padroes} padrões em {niveis} níveis "
    f"({', '.join(f'{int(100 * n)}%' for n in sorted(RESUMO.nivel.unique()))}), e a memória roda com "
    f"{v2('SELECT count(DISTINCT n_pivos) AS n FROM v2').n[0]} números de pivôs. "
    f"O arquivo bruto tem {milhar(linhas)} linhas: uma por par × padrão × nível × número de pivôs."))

## Ato 2 — O mapa estático

**Pergunta:** antes de bloquear qualquer coisa, quem guia melhor a busca?

Meço o **ganho sobre a Manhattan**: expansões da Manhattan ÷ expansões da heurística, no mesmo par. Um ganho de 3× quer dizer três vezes menos trabalho. Agrego por média geométrica, primeiro dentro de cada mapa, depois entre os mapas (o mapa é a unidade).

In [ ]:
est = PM[(PM.metrica == "absoluta") & (PM.nivel == "estatico")].pivot(index="mapa", columns="estrategia",
                                                                       values="valor")
ganho_est = 1 / est[ESTRATEGIAS]
ic_est = {e: tuple(bootstrap_bca_geom(ganho_est[e].values)) if e != "Manhattan" else (1.0, 1.0)
          for e in ESTRATEGIAS}
gm_est = {e: media_geometrica(ganho_est[e]) for e in ESTRATEGIAS}
ordem = sorted(ESTRATEGIAS[1:], key=lambda e: gm_est[e])
lider = ordem[-1]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for i, e in enumerate(ordem):
    y = ganho_est[e].sort_values().values
    x = i + np.linspace(-0.18, 0.18, len(y))
    ax.scatter(x, y, s=14, color=COR[e], alpha=0.45, marker=MARCA[e], lw=0)
    lo, hi = ic_est[e]
    ax.plot([i, i], [lo, hi], color="black", lw=1.4)
    ax.plot(i, gm_est[e], marker=MARCA[e], ms=9, color=COR[e], mec="black", mew=0.8)
    ax.text(i + 0.26, gm_est[e], f"{num(gm_est[e])}×", va="center", fontsize=9)
ax.axhline(1, color=COR["Manhattan"], lw=1, ls=":")
ax.text(-0.45, 1.04, "Manhattan = 1×", color=COR["Manhattan"], ha="left", fontsize=8.5)
ax.set_yscale("log"); ax.set_xticks(range(len(ordem)), ordem)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
ax.set_ylabel("ganho sobre a Manhattan (escala log)")
ax.set_title(f"No mapa estático, a {lider} vence: {num(gm_est[lider])}× menos expansões que a Manhattan")
ax.text(0, -0.2, f"marcador grande: média geométrica nos {N_MAPAS} mapas; barra: IC 95% BCa; pontos: um por mapa",
        transform=ax.transAxes, fontsize=8, color="0.35")
salvar(fig, "ganho_estatico")
plt.show()

In [ ]:
display(Markdown(
    f"**O que isso significa.** Sem degradação, a memória vence: a ordem é "
    f"{' > '.join(reversed(ordem))} > Manhattan. A fórmula ganha da Manhattan "
    f"({num(gm_est['Fórmula'])}×, IC {num(ic_est['Fórmula'][0])}–{num(ic_est['Fórmula'][1])}), "
    f"mas fica atrás de qualquer memória. É o ponto de partida que a degradação vai pôr à prova."))

## Ato 3 — O problema do protocolo antigo

**Pergunta:** por que o resultado do WPerformance (v1) precisava ser refeito?

No v1, o gerador de bloqueios podia cair em cima da origem ou do destino de um par, ou partir o mapa em pedaços. Os pares que ficavam sem caminho eram descartados, e os que sobravam eram escolhidos uma vez só, a 30%, e reaproveitados nos outros níveis (P2). No v2, origem e destino são protegidos (R1) e nenhum bloqueio pode dividir o mapa (R2).

In [ ]:
sob = SOB.assign(pct_v1=100 * SOB.taxa_v1, pct_v2=100 * SOB.taxa_v2)
assert (sob.pct_v2 == 100).all() and (sob.pct_v1 < 100).all()
pior = sob[sob.nivel == 0.3].sort_values("pct_v1").iloc[0]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
niv = sorted(sob.nivel.unique())
xs = [int(100 * n) for n in niv]
cinzas = dict(zip(PADRAO_PT, ["0.1", "0.27", "0.38", "0.48", "0.58"]))
marcas = dict(zip(PADRAO_PT, ["o", "s", "^", "v", "D"]))
fim = []
for p in PADRAO_PT:
    y = sob[sob.padrao == p].sort_values("nivel").pct_v1.values
    ax.plot(xs, y, color=cinzas[p], marker=marcas[p], lw=1.6)
    fim.append((p, y[-1]))
pos = 10 ** espalha(np.log10([v for _, v in fim]), 0.12)
for (p, v), yy in zip(fim, pos):
    ax.text(xs[-1] + 1, yy, f"{PADRAO_PT[p]}: {num(v)}%", va="center", fontsize=8.5, color=cinzas[p])
ax.plot(xs, [100] * len(xs), color="#009E73", lw=3, marker="o")
ax.text(xs[-1] + 1, 100, "v2, todos os padrões: 100%", va="center", fontsize=9, color="#009E73",
        fontweight="bold")
ax.text(21.5, 62, "v1 (protocolo antigo), um traço por padrão", fontsize=9, color="0.3")
ax.set_yscale("log"); ax.set_xticks(xs, [f"{x}%" for x in xs]); ax.set_xlim(xs[0] - 2, xs[-1] + 14)
ax.set_ylim(0.4, 160)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}%".replace(".", ",")))
ax.set_xlabel("células livres bloqueadas"); ax.set_ylabel("pares que continuam com caminho (escala log)")
ax.set_title("No protocolo antigo os pares sumiam com o bloqueio; no novo, todos ficam")
ax.text(0, -0.2, f"{milhar(SOB.pares_v2.iloc[0])} pares por padrão e nível, {N_MAPAS} mapas, mesmas sementes nos "
        "dois protocolos", transform=ax.transAxes, fontsize=8, color="0.35")
salvar(fig, "orientadora_sobrevivencia_v1_v2")
plt.show()

In [ ]:
pares_v2 = v2("""SELECT mapa, semente, id_problema, path_orig_manh AS caminho FROM v2
                 WHERE n_pivos = 10 AND tipo_degradacao = 'Linear' AND porcentagem_bloqueio = 0.1""")
sobrev_v1 = duckdb.sql(f"""SELECT DISTINCT mapa, semente, id_problema, tipo_degradacao, path_orig_manh AS caminho
    FROM read_csv('{MARCO / "resultados_robustez.csv"}', header = true)
    WHERE n_pivos = 10 AND porcentagem_bloqueio = 0.3""").df()
assert len(sobrev_v1) == int(SOB[SOB.nivel == 0.3].pares_v1.sum())
med_todos = pares_v2.caminho.median()
med_v1 = sobrev_v1.groupby("tipo_degradacao").caminho.median()
fig, ax = plt.subplots(figsize=(7.2, 3.8))
def ecdf(v, **kw):
    v = np.sort(np.clip(v, 1, None)); ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", **kw)
ecdf(pares_v2.caminho.values, color="#009E73", lw=2.6)
ax.text(med_todos * 1.08, 0.5, f"todos os pares (v2)\nmediana {num(med_todos, 0)}", color="#009E73", fontsize=8.5,
        va="center")
for p in PADRAO_PT:
    ecdf(sobrev_v1[sobrev_v1.tipo_degradacao == p].caminho.values, color=cinzas[p], lw=1.3)
lin = med_v1["Linear"]
ax.annotate(f"sobreviventes do v1 a 30%,\num traço por padrão\n(linear: mediana {num(lin, 0)})",
            xy=(max(lin, 1.2), 0.5), xytext=(1.4, 0.82), fontsize=8.5, color="0.3",
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xscale("log"); ax.set_xlabel("comprimento do caminho ótimo no mapa original, em células (escala log)")
ax.set_ylabel("fração dos pares")
ax.set_title("O v1 só media pares curtos: os longos eram os que sumiam")
plt.show()
display(Markdown(
    f"Mediana do caminho ótimo: **{num(med_todos, 0)} células** em todos os pares; nos sobreviventes do v1 a 30%, "
    + ", ".join(f"{PADRAO_PT[p].lower()} {num(med_v1[p], 0)}" for p in PADRAO_PT) + "."))

In [ ]:
CORES_GRID = {".": "#f6f3ec", "@": "#2b2b2b", "#": "#c9b79c", "o": "#0072B2", "r": "#CC79A7"}
def desenha_grid(ax, g, titulo, marca_passagens):
    chaves = list(CORES_GRID)
    idx = np.vectorize(chaves.index)(g)
    ax.imshow(idx, cmap=ListedColormap([CORES_GRID[c] for c in chaves]), vmin=0, vmax=len(chaves) - 1,
              interpolation="nearest")
    if marca_passagens:
        yy, xx = np.nonzero(g == "o")
        ax.scatter(xx, yy, s=2.2, color=CORES_GRID["o"], marker="o", lw=0)
    ax.set_title(titulo, fontsize=10, loc="left"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

def par_de_grids(padrao, nome_fig, titulo):
    g1, g2 = grid(f"den501d_42_{padrao}_v1_30"), grid(f"den501d_42_{padrao}_v2_30")
    r = RESUMO[(RESUMO.mapa == "den501d") & (RESUMO.semente == 42) & (RESUMO.padrao == padrao) &
               (RESUMO.nivel == 0.3)].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(9, 5.2))
    desenha_grid(axes[0], g1, f"v1: {int(r.pares_validos_v1)} de {int(r.pares_validos)} pares com caminho", False)
    desenha_grid(axes[1], g2, f"v2: {int(r.pares_validos)} de {int(r.pares_validos)} pares com caminho", True)
    leg = [Patch(color=CORES_GRID["@"], label="obstáculo original"),
           Patch(color=CORES_GRID["#"], label="bloqueada pela degradação"),
           Line2D([], [], color=CORES_GRID["o"], marker="o", ls="", ms=5,
                  label=f"passagem: recusada pela R2 ({int((g2 == 'o').sum())} células)"),
           Patch(color=CORES_GRID["r"], label=f"origem/destino protegido pela R1 ({int((g2 == 'r').sum())})")]
    fig.legend(handles=leg, loc="lower center", ncol=2, fontsize=8.5, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(titulo, x=0.02, ha="left", fontweight="bold", fontsize=11.5)
    fig.tight_layout()
    salvar(fig, nome_fig)
    plt.show()
    return r

r_lin = par_de_grids("Linear", "exemplo_linear_den501d_v1_v2",
                     "Mesmo mapa, mesma semente, linear a 30%: no v2 as paredes ganham portas")

In [ ]:
display(Markdown(
    f"**O que isso significa.** No v1, os pares que sobravam eram os fáceis: no linear a 30% sobra "
    f"{num(SOB[(SOB.padrao == 'Linear') & (SOB.nivel == 0.3)].taxa_v1.iloc[0] * 100)}% dos pares, com caminho "
    f"mediano de {num(med_v1['Linear'], 0)} células, contra {num(med_todos, 0)} no conjunto todo. No exemplo do "
    f"den501d, o v1 deixa {int(r_lin.pares_validos_v1)} pares; o v2 mantém os {int(r_lin.pares_validos)}, "
    f"abrindo portas nas paredes em vez de descartar quem ficou do outro lado. "
    f"Qualquer conclusão do v1 sobre degradação falava, na prática, de problemas curtos."))

## Ato 4 — O custo do protocolo novo

**Pergunta:** para não partir o mapa, quanto o v2 teve de deformar cada padrão?

Toda vez que um mecanismo tenta bloquear uma célula que dividiria o mapa, a R2 recusa e a célula vira passagem. Meço isso como **células recusadas pela R2 ÷ células bloqueadas**: quanto do obstáculo pretendido virou buraco.

In [ ]:
rej = REJ.assign(nome=REJ.padrao.map(PADRAO_PT))
r30 = rej[rej.nivel == 0.3].sort_values("r2_pct_bloqueadas")
top = r30.iloc[-1]
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for i, (_, linha) in enumerate(r30.iterrows()):
    vals = rej[rej.padrao == linha.padrao].sort_values("nivel")
    ax.plot(vals.r2_pct_bloqueadas, [i] * len(vals), color="0.75", lw=1.2, zorder=1)
    for (_, v), mk in zip(vals.iterrows(), ["o", "s", "D"]):
        ax.scatter(v.r2_pct_bloqueadas, i, marker=mk, s=38, color="#0072B2" if v.nivel == 0.3 else "white",
                   edgecolor="#0072B2", zorder=2)
    ax.text(vals.r2_pct_bloqueadas.max() + 1, i, f"{num(vals.r2_pct_bloqueadas.max(), 0)}%", va="center",
            fontsize=8.5)
ax.set_yticks(range(len(r30)), r30.nome)
ax.set_xlabel("células recusadas pela R2 ÷ células bloqueadas (%)")
ax.legend(handles=[Line2D([], [], marker=m, ls="", color="#0072B2", mfc=f, label=l)
                   for m, f, l in [("o", "white", "10%"), ("s", "white", "20%"), ("D", "#0072B2", "30%")]],
          title="nível", loc="lower right", fontsize=8)
ax.set_title(f"O {top.nome.lower()} é o padrão mais deformado: a 30%, {num(top.r2_pct_bloqueadas, 0)}% do que "
             "ele bloquearia vira passagem")
salvar(fig, "deformacao_por_padrao")
plt.show()

In [ ]:
r_rad = par_de_grids("Radial", "exemplo_radial_den501d_v1_v2",
                     "O radial a 30% no v2 vira queijo suíço: os círculos ganham furos para o mapa não partir")

In [ ]:
# os cinco padroes a 30% no v2, mesmo mapa e mesma semente
fig, axes = plt.subplots(1, 5, figsize=(14, 3.6))
for ax, p in zip(axes, PADRAO_PT):
    g = grid(f"den501d_42_{p}_v2_30")
    r = RESUMO[(RESUMO.mapa == "den501d") & (RESUMO.semente == 42) & (RESUMO.padrao == p) & (RESUMO.nivel == 0.3)].iloc[0]
    desenha_grid(ax, g, f"{PADRAO_PT[p]}\n{num(100 * r.pct_atingido, 1)}% bloqueado · passagens "
                        f"{num(100 * (g == 'o').sum() / (g == '#').sum(), 0)}%", True)
    ax.title.set_fontsize(9.5)
leg = [Patch(color=CORES_GRID["@"], label="obstáculo original"),
       Patch(color=CORES_GRID["#"], label="bloqueada pela degradação"),
       Line2D([], [], color=CORES_GRID["o"], marker="o", ls="", ms=5, label="passagem: recusada pela R2"),
       Patch(color=CORES_GRID["r"], label="origem/destino protegido pela R1")]
fig.legend(handles=leg, loc="lower center", ncol=4, fontsize=8.5, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("Os cinco padrões do v2 a 30% (den501d, semente 42): a meta é atingida e cada desenho ganha passagens",
             x=0.02, y=1.02, ha="left", fontweight="bold", fontsize=11.5)
fig.text(0.5, -0.1, "passagens = células recusadas pela R2 ÷ células bloqueadas pela degradação", fontsize=8.5,
         color="0.35", ha="center")
fig.tight_layout()
salvar(fig, "padroes_v2_30")
plt.show()

In [ ]:
# versao em ingles para o artigo: sem titulo e sem nota (a formula vai para a caption)
fig, axes = plt.subplots(1, 5, figsize=(14, 3.6))
for ax, p in zip(axes, PADRAO_PT):
    g = grid(f"den501d_42_{p}_v2_30")
    r = RESUMO[(RESUMO.mapa == "den501d") & (RESUMO.semente == 42) & (RESUMO.padrao == p) & (RESUMO.nivel == 0.3)].iloc[0]
    desenha_grid(ax, g, "", True)
    ax.set_title(f"{p}\n{100 * r.pct_atingido:.1f}% blocked · "
                 f"{100 * (g == 'o').sum() / (g == '#').sum():.0f} refused per 100",
                 loc="center", fontsize=9, fontweight="normal")
leg = [Patch(color=CORES_GRID["@"], label="original obstacle"),
       Patch(color=CORES_GRID["#"], label="blocked by degradation"),
       Line2D([], [], color=CORES_GRID["o"], marker="o", ls="", ms=5, label="free: refused by R2"),
       Patch(color=CORES_GRID["r"], label="start/goal protected by R1")]
fig.legend(handles=leg, loc="lower center", ncol=4, fontsize=8.5, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout()
salvar(fig, "padroes_v2_30_en")
plt.show()

In [ ]:
baixo = r30.iloc[0]
display(Markdown(
    f"**O que isso significa.** O preço de manter todos os pares é deformar os obstáculos, e o preço varia muito: "
    f"a 30%, de {num(baixo.r2_pct_bloqueadas, 0)}% no {baixo.nome.lower()} a {num(top.r2_pct_bloqueadas, 0)}% no "
    f"{top.nome.lower()}. O bloqueio total continua no alvo (a 30%, entre "
    f"{num(r30.bloqueadas_pct_S.min())}% e {num(r30.bloqueadas_pct_S.max())}% das células livres), mas um radial do "
    f"v2 não é o mesmo obstáculo de um radial do v1: é um radial que nunca fecha um corredor."))

## Ato 5 — O resultado central

**Pergunta:** quando o mapa degrada, quem conserva a vantagem sobre a Manhattan?

Este é o gráfico para levar à orientação. Ele compara o ganho sobre a Manhattan no mapa estático e no degradado (média dos três níveis e dos cinco padrões), para a fórmula e para a Memória-100 nos dois regimes.

In [ ]:
def ganho_deg(regime, e, nivel="todos", tab=G):
    q = tab[(tab.metrica == "absoluta") & (tab.regime == regime) & (tab.nivel == nivel) &
            (tab.estrategia == e)].iloc[0]
    return 1 / q.media_geometrica, 1 / q.ic95_bca_sup, 1 / q.ic95_bca_inf

linhas_ = [("Fórmula", "com_atualizacao", "Fórmula"),
           ("Memória-100", "com_atualizacao", "Memória-100 com atualização"),
           ("Memória-100", "sem_atualizacao", "Memória-100 sem atualização")]
vals = {}
for e, r, nome in linhas_:
    vals[nome] = dict(est=(gm_est[e], *ic_est[e]), deg=ganho_deg(r, e), e=e, r=r)
ret = {k: v["deg"][0] / v["est"][0] for k, v in vals.items()}
assert ret["Fórmula"] > ret["Memória-100 sem atualização"]
assert vals["Fórmula"]["deg"][0] > vals["Memória-100 sem atualização"]["deg"][0]

fig, ax = plt.subplots(figsize=(7.4, 5.2))
for nome, v in vals.items():
    st = estilo(v["e"], v["r"])
    ax.fill_between([0, 1], [v["est"][1], v["deg"][1]], [v["est"][2], v["deg"][2]], color=st["color"],
                    alpha=0.13, lw=0)
    ax.plot([0, 1], [v["est"][0], v["deg"][0]], lw=2.6, ms=8, markevery=[1], **st)
    ax.plot(0, v["est"][0], marker=st["marker"], ms=8, color=st["color"], zorder=3)
estaticos = {}
for nome, v in vals.items():
    estaticos.setdefault((v["e"], round(v["est"][0], 6)), v)
ys_e = 10 ** espalha(np.log10([v["est"][0] for v in estaticos.values()]), 0.06)
for (e, _), ye in zip(estaticos, ys_e):
    ax.text(-0.04, ye, f"{e} {num(estaticos[(e, _)]['est'][0])}×", ha="right", va="center", fontsize=10,
            color=COR[e], fontweight="bold")
ys_d = 10 ** espalha(np.log10([v["deg"][0] for v in vals.values()]), 0.06)
for (nome, v), yd in zip(vals.items(), ys_d):
    ax.text(1.04, yd, f"{num(v['deg'][0])}×  {nome}", ha="left", va="center", fontsize=10, color=COR[v["e"]],
            fontweight="bold")
ax.axhline(1, color=COR["Manhattan"], lw=1, ls=":")
ax.text(0.5, 1.05, "Manhattan = 1×", ha="center", color=COR["Manhattan"], fontsize=9)
ax.set_yscale("log"); ax.set_xlim(-0.75, 1.95)
ax.set_xticks([0, 1], ["mapa estático", "mapa degradado"]); ax.tick_params(axis="x", length=0, labelsize=10.5)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
ax.set_ylabel("ganho sobre a Manhattan (escala log)")
ax.spines["bottom"].set_visible(False)
ax.set_title("A fórmula conserva a vantagem; a memória sem atualização perde quase toda")
ax.text(0, -0.13, "Ganho = expansões da Manhattan ÷ expansões da heurística, no mesmo par e no mesmo estado do mapa.\n"
        f"Média geométrica nos {N_MAPAS} mapas (degradado: {RESUMO.padrao.nunique()} padrões × {RESUMO.nivel.nunique()} "
        "níveis com peso igual); faixas: IC 95% BCa.\nCom atualização = distâncias dos pivôs recalculadas no mapa degradado; "
        "sem = distâncias do mapa original.", transform=ax.transAxes, fontsize=8, color="0.35", va="top")
salvar(fig, "orientadora_ganho_estatico_degradado")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
xs = ["estático", "10%", "20%", "30%"]
for ax, regime in zip(axes, REGIME_PT):
    finais = []
    for e in ESTRATEGIAS[1:]:
        y = [gm_est[e]] + [ganho_deg(regime, e, n)[0] for n in ("10", "20", "30")]
        ax.plot(range(4), y, lw=2, ms=6, **estilo(e, regime))
        finais.append((e, y[-1]))
    pos = 10 ** espalha(np.log10([v for _, v in finais]), 0.055)
    for (e, v), yy in zip(finais, pos):
        ax.text(3.12, yy, f"{e.replace('Memória', 'Mem')} {num(v)}×", va="center", fontsize=8.5, color=COR[e])
    ax.axhline(1, color=COR["Manhattan"], lw=1, ls=":")
    ax.set_xticks(range(4), xs); ax.set_xlim(-0.2, 4.1); ax.set_yscale("log")
    ax.set_title(f"memória {REGIME_PT[regime]}", fontsize=10.5)
axes[0].set_ylabel("ganho sobre a Manhattan (escala log)")
axes[0].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
for n in ("10", "20", "30"):
    assert all(ganho_deg("sem_atualizacao", m, n)[0] < ganho_deg("com_atualizacao", "Fórmula", n)[0] for m in MEMORIAS)
    assert all(ganho_deg("com_atualizacao", m, n)[0] > 0.9 * gm_est[m] for m in MEMORIAS)
fig.suptitle("Com atualização a memória mantém o ganho; sem atualização ela despenca já a 10%, abaixo da fórmula",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "ganho_por_nivel")
plt.show()

In [ ]:
f, m_com, m_sem = vals["Fórmula"], vals["Memória-100 com atualização"], vals["Memória-100 sem atualização"]
display(Markdown(
    f"**O que isso significa.** A fórmula sai de {num(f['est'][0])}× para {num(f['deg'][0])}× "
    f"(conserva {num(100 * ret['Fórmula'], 0)}% do ganho). A Memória-100 com atualização vai de "
    f"{num(m_com['est'][0])}× para {num(m_com['deg'][0])}×: recalcular as distâncias mantém a memória muito à frente. "
    f"Sem atualização ela cai para {num(m_sem['deg'][0])}× (conserva {num(100 * ret['Memória-100 sem atualização'], 0)}%), "
    f"abaixo da fórmula. A fórmula não guarda nada do mapa, então não tem nada a ficar desatualizado."))

## Ato 6 — Por que o δ engana

**Pergunta:** se a fórmula vence sem atualização, por que a Manhattan aparece como "a menos degradada"?

O δ é a perda relativa de cada heurística: expansões no mapa degradado ÷ expansões no mapa original, **dela mesma**. Pense em dois motoristas numa cidade que fecha ruas. Um não conhece a cidade e já dava voltas enormes: com as ruas fechadas, ele dá voltas um pouco maiores. O outro conhecia todos os atalhos: quando os atalhos fecham, a viagem dele piora muito em proporção, mas ele **continua chegando antes**. O δ do primeiro é menor, e mesmo assim ele é o pior motorista. A Manhattan é o primeiro motorista: não tem informação do mapa a perder, então o δ dela mede só o quanto o problema ficou mais difícil.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metrica, titulo in ((axes[0], "delta", "δ: perda relativa (menor = perdeu menos)"),
                            (axes[1], "absoluta", "ganho sobre a Manhattan no mapa degradado")):
    for i, e in enumerate(ESTRATEGIAS):
        for regime, dx in (("com_atualizacao", -0.12), ("sem_atualizacao", 0.12)):
            if e not in MEMORIAS and regime == "sem_atualizacao":
                continue
            q = G[(G.metrica == metrica) & (G.regime == regime) & (G.nivel == "todos") & (G.estrategia == e)].iloc[0]
            v, lo, hi = q.media_geometrica, q.ic95_bca_inf, q.ic95_bca_sup
            if metrica == "absoluta":
                v, lo, hi = 1 / v, 1 / hi, 1 / lo
            x = i + (dx if e in MEMORIAS else 0)
            st = estilo(e, regime)
            if np.isfinite(lo):
                ax.plot([x, x], [lo, hi], color=st["color"], lw=1.2, ls=st["ls"])
            ax.plot(x, v, marker=st["marker"], ms=8, color=st["color"], mfc=st["markerfacecolor"], mec=st["color"])
    ax.set_xticks(range(len(ESTRATEGIAS)), [e.replace("Memória", "Mem") for e in ESTRATEGIAS], fontsize=8.5)
    ax.set_yscale("log"); ax.set_title(titulo, fontsize=10.5)
    suf = "×" if metrica == "absoluta" else ""
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _, suf=suf: f"{v:g}{suf}".replace(".", ",")))
    ax.yaxis.set_minor_formatter(mpl.ticker.FuncFormatter(lambda v, _, suf=suf: f"{v:g}{suf}".replace(".", ",")
                                                          if v in (2, 3, 4, 6) else ""))
axes[0].set_ylabel("δ = expansões degradado ÷ original (escala log)")
axes[1].set_ylabel("ganho (escala log)")
axes[1].legend(handles=[Line2D([], [], marker="o", color=COR["Memória-100"], label="memória com atualização"),
                        Line2D([], [], marker="o", color=COR["Memória-100"], mfc="white", ls="--",
                               label="memória sem atualização")], fontsize=8, loc="upper left")
d_manh = G[(G.metrica == "delta") & (G.nivel == "todos") & (G.estrategia == "Manhattan")].media_geometrica.iloc[0]
d_form = G[(G.metrica == "delta") & (G.nivel == "todos") & (G.estrategia == "Fórmula")].media_geometrica.iloc[0]
assert d_manh < d_form and ganho_deg("sem_atualizacao", "Fórmula")[0] > 1
fig.suptitle("A Manhattan perde menos (δ) porque não tem o que perder; em expansões ela continua a pior",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "delta_contra_ganho")
plt.show()

In [ ]:
display(Markdown(
    f"**O que isso significa.** No δ, a Manhattan ({num(d_manh, 3)}) parece mais robusta que a fórmula "
    f"({num(d_form, 3)}). Mas, no mesmo mapa degradado, a fórmula ainda faz {num(ganho_deg('sem_atualizacao', 'Fórmula')[0])}× "
    f"menos expansões que a Manhattan. Por isso o artigo usa as duas métricas: o δ diz quanto cada heurística perdeu; o "
    f"ganho diz quem continua guiando melhor."))

## Ato 7 — Consistência

**Pergunta:** o resultado vale mapa a mapa, ou é efeito da média?

In [ ]:
def matriz(pm, regime):
    t = pm[(pm.metrica == "absoluta") & (pm.nivel == "todos") & (pm.regime == regime)]
    t = 1 / t.pivot(index="mapa", columns="estrategia", values="valor")[ESTRATEGIAS[1:]]
    return t.loc[sorted(t.index, key=lambda m: (GRUPOS.index(categoria(m)), m))]

def heat(ax, t, titulo):
    lv = np.log10(t.values)
    lim = np.abs(lv).max()
    ax.imshow(lv, cmap="PuOr", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
    melhor = t.values.argmax(axis=1)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            txt = f"{num(v, 1)}" + (" ★" if j == melhor[i] else "") + (" ✗" if v < 1 else "")
            ax.text(j, i, txt, ha="center", va="center", fontsize=7.5,
                    fontweight="bold" if j == melhor[i] else "normal",
                    color="white" if abs(lv[i, j]) > 0.62 * lim else "black")
    ax.set_xticks(range(t.shape[1]), [c.replace("Memória", "Mem") for c in t.columns], fontsize=8)
    ax.set_yticks(range(t.shape[0]), t.index, fontsize=8)
    ax.tick_params(length=0); ax.set_title(titulo, fontsize=10)
    for s in ax.spines.values(): s.set_visible(False)

t_todos, t_fora = matriz(PM, "sem_atualizacao"), matriz(PMF, "sem_atualizacao")
v_todos = int((t_todos.idxmax(axis=1) == "Fórmula").sum()); v_fora = int((t_fora.idxmax(axis=1) == "Fórmula").sum())
fig, axes = plt.subplots(1, 2, figsize=(10, 6.2))
heat(axes[0], t_todos, f"todos os pares: fórmula melhor em {v_todos} de {len(t_todos)}")
heat(axes[1], t_fora, f"só pares fora do treino do AG: {v_fora} de {len(t_fora)}")
fig.suptitle("Sem atualização, a fórmula é a melhor heurística em todos os mapas (ganho sobre a Manhattan)",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.text(0.02, -0.01, "★ melhor do mapa   ✗ pior que a Manhattan (ganho < 1)   roxo: ganha da Manhattan; laranja: perde",
         fontsize=8.5, color="0.35")
fig.tight_layout()
salvar(fig, "consistencia_por_mapa")
plt.show()

In [ ]:
perde = [(m, e) for m in t_todos.index for e in MEMORIAS if t_todos.loc[m, e] < 1]
col_mapas = sorted(COLAPSO.mapa.unique())
ligados = sorted({m for m, _ in perde} & set(col_mapas))
desc = "; ".join(f"{m} ({e}, {num(t_todos.loc[m, e])}×)" for m, e in perde) or "nenhum"
display(Markdown(
    f"Casos em que a memória sem atualização perde para a Manhattan: {desc}. "
    f"Os mapas com colapso de pivô em alguma semente (P8: o primeiro pivô cai num bolsão isolado e a memória vira "
    f"h = 0) são: {', '.join(col_mapas)}. "
    + (f"Entre os casos acima, {', '.join(ligados)} {'está' if len(ligados) == 1 else 'estão'} nessa lista."
       if ligados else "Nenhum dos casos acima está nessa lista.")))
display(COLAPSO)

In [ ]:
display(Markdown(
    f"**O que isso significa.** Não é efeito da média: sem atualização, a fórmula é a melhor heurística em "
    f"{v_todos} de {len(t_todos)} mapas, e o resultado se repete ({v_fora} de {len(t_fora)}) quando eu tiro os pares que o "
    f"algoritmo genético viu durante a síntese. Então a fórmula não está só decorando os pares de treino."))

## Ato 8 — A inversão dos postos

**Pergunta:** a ordem das heurísticas no mapa estático se mantém quando o mapa degrada?

O posto médio de Friedman ordena as seis configurações dentro de cada mapa (1 = melhor) e tira a média entre os mapas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, metrica, titulo in ((axes[0], "delta", "δ (perda relativa)"), (axes[1], "absoluta", "ganho absoluto")):
    q = POSTOS[(POSTOS.metrica == metrica) & (POSTOS.regime == "sem_atualizacao") & (POSTOS.nivel == "todos")]
    q = q.set_index("estrategia").loc[ESTRATEGIAS]
    for e, lin in q.iterrows():
        st = estilo(e, "sem_atualizacao")
        ax.plot([0, 1], [lin.posto_estatico, lin.posto_degradado], lw=2.2, ms=7, **st)
    pe = espalha(q.posto_estatico.values, 0.28); pd_ = espalha(q.posto_degradado.values, 0.28)
    for (e, lin), a, b in zip(q.iterrows(), pe, pd_):
        ax.text(-0.05, a, f"{e.replace('Memória', 'Mem')} {num(lin.posto_estatico)}", ha="right", va="center",
                fontsize=8.5, color=COR[e])
        ax.text(1.05, b, f"{num(lin.posto_degradado)} {e.replace('Memória', 'Mem')}", ha="left", va="center",
                fontsize=8.5, color=COR[e])
    ax.set_ylim(6.5, 0.5); ax.set_xlim(-0.7, 1.7)
    ax.set_xticks([0, 1], ["estático", "degradado,\nsem atualização"]); ax.set_yticks([])
    for s in ("left", "bottom"): ax.spines[s].set_visible(False)
    ax.tick_params(length=0); ax.set_title(titulo, fontsize=10.5)
qd = POSTOS[(POSTOS.metrica == "delta") & (POSTOS.regime == "sem_atualizacao") & (POSTOS.nivel == "todos")]
rho = qd.posto_estatico.rank().corr(qd.posto_degradado.rank())
inverte = bool(np.isclose(rho, -1))
fig.suptitle("Sem atualização, a ordem estática se inverte" + (" por completo no δ" if inverte else ""),
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "inversao_postos")
plt.show()
qa = POSTOS[(POSTOS.metrica == "absoluta") & (POSTOS.regime == "sem_atualizacao") & (POSTOS.nivel == "todos")]
qa = qa.set_index("estrategia")
display(Markdown(
    f"No δ, a correlação de postos entre a ordem estática e a degradada sem atualização é {num(rho)}"
    + (": a ordem vira exatamente do avesso." if inverte else ".")
    + f" No ganho absoluto, a fórmula sai do posto {num(qa.posto_estatico['Fórmula'])} para "
    f"{num(qa.posto_degradado['Fórmula'])}, e a Memória-100 do {num(qa.posto_estatico['Memória-100'])} para "
    f"{num(qa.posto_degradado['Memória-100'])}."))

Os diagramas de diferença crítica (Nemenyi) que o pipeline já gera: (a) estático, (b) degradado com atualização, (c) degradado sem atualização. Configurações ligadas por uma barra não se separam estatisticamente.

In [ ]:
for nome in ("fig_b1_cd_absoluta", "fig_b1_cd_delta"):
    display(Markdown(f"`figuras_v2/{nome}.png`"))
    display(Image(filename=str(ANALISE / "figuras_v2" / f"{nome}.png"), width=520))

## Ato 9 — O padrão não troca a ordem

**Pergunta:** a forma do bloqueio (linear, radial…) muda quem ganha de quem?

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(11, 3.4), sharey=True)
for ax, p in zip(axes, PADRAO_PT):
    for e, r in (("Fórmula", "com_atualizacao"), ("Memória-100", "com_atualizacao"), ("Memória-100", "sem_atualizacao")):
        q = PN[(PN.metrica == "absoluta") & (PN.regime == r) & (PN.padrao == p) & (PN.estrategia == e)]
        q = q.sort_values("nivel")
        ax.plot(q.nivel.astype(int), 1 / q.valor, lw=1.8, ms=5, **estilo(e, r))
    ax.axhline(1, color=COR["Manhattan"], lw=0.8, ls=":")
    ax.set_title(PADRAO_PT[p], fontsize=10); ax.set_yscale("log")
    nv = sorted(PN.nivel.astype(int).unique())
    ax.set_xticks(nv, [f"{n}%" for n in nv], fontsize=8)
axes[0].set_ylabel("ganho sobre a Manhattan (log)")
pa = PN[PN.metrica == "absoluta"]
for (p, n), q in pa.groupby(["padrao", "nivel"]):
    v = q.set_index(["regime", "estrategia"]).valor
    assert v[("com_atualizacao", "Memória-100")] < v[("com_atualizacao", "Fórmula")] < v[("sem_atualizacao", "Memória-100")]
axes[0].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
fig.legend(handles=[Line2D([], [], lw=1.8, ms=5, label=rotulo(e, r), **estilo(e, r))
                    for e, r in (("Fórmula", "com_atualizacao"), ("Memória-100", "com_atualizacao"),
                                 ("Memória-100", "sem_atualizacao"))],
           loc="lower center", ncol=3, fontsize=8.5, bbox_to_anchor=(0.5, -0.08))
fig.suptitle("A ordem é a mesma nos cinco padrões; o que muda é o nível de bloqueio", x=0.02, ha="left",
             fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "padroes_facetas")
plt.show()

In [ ]:
termos = {"C(padrao)": "padrão", "C(nivel)": "nível", "C(estrategia):C(padrao)": "estratégia × padrão",
          "C(estrategia):C(nivel)": "estratégia × nível"}
a2v = ANOVA_V2[("delta", "com_atualizacao")]
eta = pd.DataFrame({"v1 (WPerformance)": ANOVA_V1.loc[list(termos), "eta2_parcial"],
                    "v2": a2v.loc[list(termos), "eta2_parcial"]}).rename(index=termos)
fig, ax = plt.subplots(figsize=(7.2, 2.9))
for i, (t, lin) in enumerate(eta.iterrows()):
    ax.plot([lin.iloc[0], lin.iloc[1]], [i, i], color="0.75", lw=1.5)
    ax.scatter(lin.iloc[0], i, color="white", edgecolor="0.3", s=50, zorder=3, marker="s")
    ax.scatter(lin.iloc[1], i, color="#009E73", s=55, zorder=3)
    ax.text(max(lin) + 0.012, i, f"{num(lin.iloc[0], 3)} → {num(lin.iloc[1], 3)}", va="center", fontsize=8.5)
ax.set_yticks(range(len(eta)), eta.index); ax.invert_yaxis()
ax.set_xlabel("η²ₚ (log δ, memória com atualização, como no v1)")
ax.legend(handles=[Line2D([], [], marker="s", ls="", mfc="white", mec="0.3", label="v1"),
                   Line2D([], [], marker="o", ls="", color="#009E73", label="v2")], loc="lower right", fontsize=8)
ax.set_xlim(0, eta.values.max() + 0.12)
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",")))
ax.set_title("Com a conectividade preservada, o padrão quase não muda a ordem das estratégias")
salvar(fig, "eta2_v1_v2")
plt.show()
it = INTER.set_index(["metrica", "regime"])
display(it[["F", "eta2_parcial", "p_permutacao", "n_permutacoes", "eta2_parcial_estrategia_nivel"]].round(4))

In [ ]:
e1, e2 = eta["v1 (WPerformance)"], eta["v2"]
pesa = "mais" if e2["nível"] > e2["padrão"] else "menos"
display(Markdown(
    f"**O que isso significa.** A interação estratégia × padrão cai de η²ₚ = {num(e1['estratégia × padrão'], 3)} no v1 "
    f"para {num(e2['estratégia × padrão'], 3)} no v2: boa parte do \"efeito do padrão\" no v1 vinha de cada padrão "
    f"descartar pares diferentes. No v2, o nível de bloqueio pesa {pesa} que o padrão sobre o δ "
    f"(η²ₚ {num(e2['nível'], 3)} contra {num(e2['padrão'], 3)}; no v1 era {num(e1['nível'], 3)} contra "
    f"{num(e1['padrão'], 3)}). A interação continua estatisticamente detectável (p de permutação "
    f"{num(it.loc[('delta', 'com_atualizacao'), 'p_permutacao'], 3)}, o piso com "
    f"{int(it.loc[('delta', 'com_atualizacao'), 'n_permutacoes'])} permutações). O padrão muda o tamanho das "
    f"diferenças, mas não troca a ordem: nas facetas, a fórmula fica entre a memória com e a sem atualização em todos "
    f"os padrões e níveis. O ato 13 refaz este teste com o mapa como unidade."))

## Ato 10 — O preço da fórmula: caminhos mais longos

**Pergunta:** a fórmula não é admissível. Ela expande menos, mas quanto mais longos ficam os caminhos que ela acha, principalmente no mapa modificado?

Meço **ρ = caminho da fórmula ÷ caminho ótimo**. No mapa degradado, o ótimo é o caminho da Manhattan, que é admissível; antes de usar, conferi contra uma BFS (Dijkstra de custo unitário) no próprio mapa degradado.

In [ ]:
le2 = lambda nome, pasta=T2: pd.read_csv(pasta / f"{nome}.csv")
RHO, RHOPM = le2("b2_rho").set_index("cenario"), le2("b2_rho_por_mapa").set_index("mapa")
CONF, PA, FRASE2 = le2("b2_conferencia_otimo").iloc[0], le2("b2_proposicao_a"), le2("b2_frase").iloc[0]
RHO_F = le2("b2_rho", T2F).set_index("cenario")
rd, rs = RHO.loc["degradado (todos)"], RHO.loc["estático"]
mapas_ = sorted(RHOPM.index, key=lambda m: (GRUPOS.index(categoria(m)), m))
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for i, m in enumerate(mapas_):
    a, b = RHOPM.loc[m, "rho_estatico"], RHOPM.loc[m, "rho_degradado"]
    ax.plot([a, b], [i, i], color="0.8", lw=1.4, zorder=1)
    ax.scatter(a, i, color="white", edgecolor=COR["Fórmula"], s=30, zorder=2, marker="D")
    ax.scatter(b, i, color=COR["Fórmula"], s=30, zorder=3, marker="D")
ax.set_yticks(range(len(mapas_)), mapas_, fontsize=8); ax.invert_yaxis()
ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(0.05))
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{100 * (v - 1):+.0f}%".replace(".", ",")))
ax.set_xlabel("excesso de caminho da fórmula sobre o ótimo (média geométrica de ρ − 1, por mapa)")
ax.legend(handles=[Line2D([], [], marker="D", ls="", mfc="white", mec=COR["Fórmula"], label="mapa estático"),
                   Line2D([], [], marker="D", ls="", color=COR["Fórmula"], label="mapa degradado")],
          loc="lower right", fontsize=8.5)
ax.set_title(f"Os caminhos da fórmula ficam {num(100 * (rd.rho_media_geometrica - 1), 1)}% mais longos no mapa "
             f"degradado (eram {num(100 * (rs.rho_media_geometrica - 1), 1)}% no estático)")
salvar(fig, "rho_formula_por_mapa")
plt.show()
viol = PA[["viol_original", "viol_com_atualizacao", "viol_sem_atualizacao"]].to_numpy().sum()
display(Markdown(
    f"**Referência ótima.** Em {milhar(CONF.pares_conferidos)} pares conferidos contra a BFS no mapa degradado, o "
    f"caminho da Manhattan foi o ótimo em {milhar(CONF.iguais)} ({milhar(CONF.diferentes)} diferenças).\n\n"
    f"**ρ da fórmula.** No estático, {num(100 * rs.frac_otimos, 1)}% dos caminhos são ótimos e "
    f"{num(100 * rs.frac_excesso_10pct, 1)}% passam de 10% de excesso; no degradado, "
    f"{num(100 * rd.frac_otimos, 1)}% e {num(100 * rd.frac_excesso_10pct, 1)}%. Média geométrica nos mapas: "
    f"{num(rs.rho_media_geometrica, 3)} [IC {num(rs.ic95_bca_inf, 3)}–{num(rs.ic95_bca_sup, 3)}] no estático e "
    f"{num(rd.rho_media_geometrica, 3)} [IC {num(rd.ic95_bca_inf, 3)}–{num(rd.ic95_bca_sup, 3)}] no degradado "
    + "(por nível: " + ", ".join(f"{c.split()[-1]} {num(RHO.loc[c, 'rho_media_geometrica'], 3)}"
                                 for c in RHO.index if c.startswith("degradado ") and "%" in c) + "). "
    f"Só com os pares fora do treino: {num(RHO_F.loc['degradado (todos)', 'rho_media_geometrica'], 3)}.\n\n"
    f"**Proposição A.** A memória continua admissível no mapa degradado: em {milhar(PA.pares.sum())} buscas por "
    f"regime, o caminho dela foi diferente do ótimo em {milhar(viol)} casos, com e sem atualização.\n\n"
    f"**A frase.** Sem atualização, a fórmula expande {num(FRASE2.expansoes_x)}× menos que a melhor memória "
    f"congelada ({FRASE2.melhor_memoria_congelada}), ao custo de caminhos {num(FRASE2.excesso_caminho_pct, 1)}% mais "
    f"longos em média."))

## Ato 11 — Por que a memória congelada perde

**Pergunta:** o que faz a memória sem atualização perder tanto?

A hipótese: a tabela congelada guarda as distâncias do mapa antigo, então ela erra mais quanto mais o caminho mudou. Para cada par, meço o **desvio r** = caminho ótimo no mapa degradado ÷ caminho ótimo no original (r = 1: o caminho não mudou) e o **aumento de expansões** de cada heurística (degradado ÷ original).

In [ ]:
FX4 = le2("b4_aumento_por_faixa")
SP4 = le2("b4_spearman_resumo").set_index("heuristica")
W4 = le2("b4_wilcoxon")
R14 = le2("b4_caso_r1").set_index("heuristica")
FAIXAS = ["r = 1", "1–1,1", "1,1–1,5", "1,5–2", "> 2"]
linhas4 = [("Manhattan", "Manhattan", "com_atualizacao"), ("Fórmula", "Fórmula", "com_atualizacao"),
           ("Memória-100 com atualização", "Memória-100", "com_atualizacao"),
           ("Memória-100 sem atualização", "Memória-100", "sem_atualizacao")]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(len(FAIXAS))
fins = []
for nome, e, r in linhas4:
    q = FX4[FX4.heuristica == nome].set_index("faixa").reindex(FAIXAS)
    st = estilo(e, r)
    ax.fill_between(x, q.q1, q.q3, color=st["color"], alpha=0.12, lw=0)
    ax.plot(x, q.mediana, lw=2, ms=6, **st)
    fins.append((nome, q.mediana.iloc[-1], st["color"]))
pos = 10 ** espalha(np.log10([v for _, v, _ in fins]), 0.07)
for (nome, v, c), y in zip(fins, pos):
    ax.text(x[-1] + 0.12, y, f"{nome.replace('Memória', 'Mem')} {num(v, 1)}×", va="center", fontsize=8.5, color=c)
ax.set_yscale("log"); ax.set_xticks(x, FAIXAS); ax.set_xlim(-0.2, len(FAIXAS) + 0.9)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
ax.set_xlabel("desvio r = caminho ótimo degradado ÷ original")
ax.set_ylabel("aumento de expansões (escala log)")
salto = FX4[(FX4.heuristica == "Memória-100 sem atualização") & (FX4.faixa == "1–1,1")].mediana.iloc[0]
salto_f = FX4[(FX4.heuristica == "Fórmula") & (FX4.faixa == "1–1,1")].mediana.iloc[0]
ax.set_title(f"A memória congelada perde em degrau: com até 10% de desvio, já expande {num(salto, 1)}× mais "
             f"(a fórmula, {num(salto_f)}×)")
ax.text(0, -0.17, "Mediana entre os mapas das medianas por mapa; faixa: 1º a 3º quartil entre os mapas.",
        transform=ax.transAxes, fontsize=8, color="0.35")
salvar(fig, "mecanismo_desvio")
plt.show()
w = W4[(W4.memoria == "Memória-100 sem atualização") & (W4.referencia == "Fórmula")].iloc[0]
w10 = W4[(W4.memoria == "Memória-10 sem atualização") & (W4.referencia == "Fórmula")].iloc[0]
sp = lambda h: f"{num(SP4.loc[h, 'mediana'])} [{num(SP4.loc[h, 'minimo'])}; {num(SP4.loc[h, 'maximo'])}]"
display(Markdown(
    f"**Correlação de Spearman entre r e o aumento**, por mapa (mediana [mín; máx] nos {int(SP4.n_mapas.iloc[0])} "
    f"mapas): Manhattan {sp('Manhattan')}, fórmula {sp('Fórmula')}, Memória-100 sem atualização "
    f"{sp('Memória-100 sem atualização')}, Memória-100 com atualização {sp('Memória-100 com atualização')}. "
    f"A Memória-100 congelada tem correlação maior que a fórmula em {int(w.mapas_memoria_maior)} de "
    f"{int(w.n_mapas)} mapas (Wilcoxon p = {num(w.wilcoxon_p, 4)}; com Holm sobre as {len(W4)} comparações, "
    f"{num(w.wilcoxon_p_holm, 3)}); a Memória-10 congelada, em "
    f"{int(w10.mapas_memoria_maior)} de {int(w10.n_mapas)} (p = {num(w10.wilcoxon_p, 2)}; Holm {num(w10.wilcoxon_p_holm, 2)}).\n\n"
    f"**Quando o caminho ótimo não muda (r = 1)**, o aumento mediano é "
    + ", ".join(f"{h.replace('Memória', 'Mem')} {num(R14.loc[h, 'mediana'])}×" for h, _, _ in linhas4)
    + f"; a fração de pares que piora fica entre {num(100 * R14.frac_aumento_maior_1.min(), 0)}% e "
      f"{num(100 * R14.frac_aumento_maior_1.max(), 0)}% em todas as heurísticas.\n\n"
    f"**O que isso significa.** A correlação sozinha não separa bem as heurísticas: todas pioram quando o caminho "
    f"desvia. O que separa é a **magnitude**: assim que o caminho muda um pouco, a tabela congelada deixa de guiar "
    f"a busca e a memória perde quase tudo de uma vez, enquanto a fórmula e a Manhattan, que não guardam nada do "
    f"mapa, pioram aos poucos. Se o caminho não muda, ninguém perde, nem a memória congelada."))

## Ato 12 — Quando vale a pena atualizar a memória

**Pergunta:** se o mapa muda de vez em quando, a partir de quantas buscas entre duas mudanças compensa recalcular a memória?

Conto o custo em operações, não em tempo (P3). Atualizar a memória custa as células visitadas pelas BFS dos pivôs no mapa novo (contadas na medição, conferidas contra a BFS real). Cada busca custa as suas expansões. Supondo que 1 expansão do A\* custa o mesmo que *c* visitas de BFS, o custo de um intervalo com Q buscas é Q·c·(expansões por busca), mais o custo de atualizar, para a memória atualizada. O ponto de equilíbrio Q\* é onde a memória atualizada passa a custar menos que a alternativa.

In [ ]:
EQ = le2("b3_equilibrio_por_mapa")
EQS = le2("b3_equilibrio_resumo")
EQB = le2("b3_equilibrio_melhor_memoria")
SE = le2("b3_sem_economia")
FRASE3 = le2("b3_frase").iloc[0]
qs = np.logspace(0, 6, 300)
def curvas(maps, n=100, c=1):
    t = EQ[(EQ.alternativa == "formula") & (EQ.n_pivos == n) & (EQ.c == c) & EQ.mapa.isin(maps)].groupby("mapa").agg(
        u=("custo_atualizar", "mean"), eu=("exp_mem_atualizada", "mean"), ef=("exp_alternativa", "mean"),
        em=("exp_manhattan", "mean"))
    ec = EQ[(EQ.alternativa == "sem_atualizacao") & (EQ.n_pivos == n) & (EQ.c == c) & EQ.mapa.isin(maps)] \
        .groupby("mapa").exp_alternativa.mean().reindex(t.index)
    cur = {("Manhattan", "com_atualizacao"): np.outer(np.ones(len(t)), qs),
           ("Fórmula", "com_atualizacao"): np.outer(t.ef / t.em, qs),
           (f"Memória-{n}", "sem_atualizacao"): np.outer(ec / t.em, qs),
           (f"Memória-{n}", "com_atualizacao"): np.outer(t.eu / t.em, qs) + (t.u / (c * t.em)).values[:, None]}
    return {k: np.median(v, axis=0) for k, v in cur.items()}

def painel(ax, cur, rotulos=True):
    pos = 10 ** espalha(np.log10([v[-1] for v in cur.values()]), 0.3)
    for ((e, r), v), y in zip(cur.items(), pos):
        st = estilo(e, r)
        ax.plot(qs, v, lw=2, color=st["color"], ls=st["ls"] if e != "Manhattan" else ":", alpha=st["alpha"])
        if rotulos:
            ax.text(qs[-1] * 1.3, y, rotulo(e, r).replace("Memória", "Mem"), va="center", fontsize=8.5,
                    color=st["color"])
    ax.set_xscale("log"); ax.set_yscale("log")

cur = curvas(EQ.mapa.unique())
cruz = qs[np.argmax(cur[("Memória-100", "com_atualizacao")] < cur[("Fórmula", "com_atualizacao")])]
fig, ax = plt.subplots(figsize=(7.4, 4.6))
painel(ax, cur)
ax.axvline(cruz, color="0.6", lw=0.8, ls="-.")
ax.text(cruz * 1.15, cur[("Manhattan", "com_atualizacao")][0] * 2,
        f"cruzamento das curvas\ndo mapa mediano ≈ {milhar(round(cruz, -1))}", fontsize=8.5, color="0.3")
ax.set_xlim(1, qs[-1] * 40)
ax.set_xlabel("buscas entre duas mudanças do mapa, Q (escala log)")
ax.set_ylabel("custo do intervalo, em buscas da Manhattan (escala log)")
ax.set_title(f"Abaixo de ~{milhar(FRASE3.q_estrela_mediana)} buscas por mudança, a fórmula custa menos; acima, "
             "atualizar a memória compensa")
ax.text(0, -0.15, f"Curvas: mediana entre os mapas do custo normalizado pela busca da Manhattan; c = 1; Memória-100; "
        f"todos os níveis.\nTítulo: mediana dos Q* calculados mapa a mapa ({milhar(FRASE3.q_estrela_mediana)}); "
        f"as duas medianas não precisam coincidir.", transform=ax.transAxes, fontsize=8, color="0.35", va="top")
salvar(fig, "equilibrio_atualizacao")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(GRUPOS), figsize=(11, 3.2), sharey=True)
for ax, g in zip(axes, GRUPOS):
    ms = [m for m in EQ.mapa.unique() if categoria(m) == g]
    c_ = curvas(ms)
    painel(ax, c_, rotulos=False)
    x_ = qs[np.argmax(c_[("Memória-100", "com_atualizacao")] < c_[("Fórmula", "com_atualizacao")])]
    ax.axvline(x_, color="0.6", lw=0.8, ls="-.")
    ax.set_title(f"{g} ({len(ms)} mapas): Q* ≈ {milhar(round(x_, -1))}", fontsize=9.5)
    ax.set_xlabel("Q (log)")
axes[0].set_ylabel("custo do intervalo (buscas Manhattan)")
fig.legend(handles=[Line2D([], [], lw=2, color=estilo(e, r)["color"], ls=estilo(e, r)["ls"] if e != "Manhattan" else ":",
                           label=rotulo(e, r)) for e, r in [("Fórmula", "com_atualizacao"), ("Memória-100", "com_atualizacao"),
                                                           ("Memória-100", "sem_atualizacao"), ("Manhattan", "com_atualizacao")]],
           loc="lower center", ncol=4, fontsize=8.5, bbox_to_anchor=(0.5, -0.1))
fig.suptitle("O ponto de equilíbrio muda de perfil para perfil, mas a forma é a mesma", x=0.02, ha="left",
             fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "equilibrio_por_perfil")
plt.show()

In [ ]:
t = EQS[(EQS.alternativa == "formula") & (EQS.n_pivos == 100)].copy()
t["Q* (mediana [mín; máx])"] = t.apply(lambda r: f"{milhar(r.mediana)} [{milhar(r.minimo)}; {milhar(r.maximo)}]", axis=1)
t["nível"] = (100 * t.nivel).astype(int).astype(str) + "%"
display(Markdown("**Sensibilidade a c** (Memória-100 contra a fórmula):"))
display(t.pivot(index="c", columns="nível", values="Q* (mediana [mín; máx])"))
sa = EQS[(EQS.alternativa == "sem_atualizacao") & (EQS.c == 1)]
semeco = int(EQS[EQS.alternativa == "formula"].mapas_sem_economia.max())
display(Markdown(
    f"Q\* escala exatamente com 1/c: dobrar o custo de uma expansão divide o ponto de equilíbrio por dois. Com c entre "
    f"1 e 10, a mediana (Memória-100, todos os níveis) vai de {milhar(FRASE3.q_estrela_mediana)} a "
    f"{milhar(FRASE3.q_estrela_mediana / 10)} buscas. **A conclusão não muda com c**: continua existindo um limiar, "
    f"abaixo dele a fórmula é a melhor escolha e acima dele atualizar compensa; c só desloca o limiar. "
    f"Contra a memória sem atualização (mesmos pivôs), atualizar compensa "
    + ("ainda mais cedo" if sa.mediana.max() < t[t.c == 1].mediana.min() else "a partir de")
    + f": mediana entre {milhar(sa.mediana.min())} e {milhar(sa.mediana.max())} buscas com c = 1, conforme o nível e "
    f"o número de pivôs. "
    + "\n\n"
    + (f"**Sem economia.** Em {len(SE)} combinações (mapa, nível, pivôs) a memória atualizada expande mais que a "
       f"fórmula e nunca compensa: " + ", ".join(sorted({f"{r.mapa} com {r.n_pivos} pivôs" for r in SE.itertuples()}))
       + ", exatamente as células de colapso de pivô (ato 14).\n\n" if len(SE) else "")
    + f"**A memória mais barata muda o limiar.** Escolhendo em cada mapa o número de pivôs que alcança a fórmula com "
    f"menos buscas (em geral 10 ou 20), o limiar com c = 1 cai para "
    + ", ".join(f"{milhar(r.mediana)} [{milhar(r.minimo)}; {milhar(r.maximo)}] a {int(100 * r.nivel)}%"
                for r in EQB[EQB.c == 1].itertuples())
    + f" buscas; com c = 10, para "
    + ", ".join(f"{milhar(r.mediana)}" for r in EQB[EQB.c == 10].itertuples())
    + ".\n\n"
    f"**A frase.** {FRASE3.frase}\n\n"
    f"**O que isso significa.** Quando a memória pode ser atualizada, a fórmula só é a melhor escolha se o mapa muda "
    f"com frequência: a cada poucas centenas de buscas com a Memória-100, ou a cada pouco mais de cem com uma memória "
    f"pequena. Abaixo disso, a fórmula vence porque não paga atualização nenhuma; acima, a memória atualizada "
    f"amortiza o custo."))

## Ato 13 — A estatística sem pseudorreplicação

**Pergunta:** a interação estratégia × padrão (ato 9) continua de pé quando eu trato o mapa como a unidade?

O teste do ato 9 usa células mapa × semente × padrão × nível como se fossem independentes, mas elas vêm de poucos mapas. Aqui o mesmo efeito é testado de três jeitos: modelo misto (intercepto aleatório por mapa e variância da semente dentro do mapa, teste de razão de verossimilhança), modelo misto ampliado (mais uma variância da estratégia dentro do mapa) e ANOVA de medidas repetidas com o mapa como sujeito.

In [ ]:
MM = le2("b5_modelo_misto")
nome_mod = {"misto_padrao": "misto", "misto_ampliado": "misto ampliado", "anova_rm_mapa": "medidas repetidas",
            "ols_permutacao_b1": "permutação (ato 9)"}
MM["teste"] = MM.modelo.map(nome_mod)
MM["p"] = MM.p.map(lambda p: "< 0,001" if p < 0.001 else num(p, 3))
MM["caso"] = MM.metrica.map({"delta": "δ", "absoluta": "ganho"}) + ", " + MM.regime.map(REGIME_PT)
display(MM.pivot(index="caso", columns="teste", values="p")[list(nome_mod.values())])
v = le2("b5_modelo_misto")
v = v[v.modelo == "misto_padrao"]
rm = le2("b5_modelo_misto")
rm = rm[rm.modelo == "anova_rm_mapa"]
conc = (le2("b5_modelo_misto").groupby(["metrica", "regime"]).p.apply(lambda p: (p < 0.05).all())).all()
display(Markdown(
    f"Variância do mapa entre {num(v.var_mapa.min(), 3)} e {num(v.var_mapa.max(), 3)}, da semente entre "
    f"{num(v.var_semente.min(), 4)} e {num(v.var_semente.max(), 4)}, resíduo entre {num(v.var_residuo.min(), 3)} e "
    f"{num(v.var_residuo.max(), 3)} (escala log). Na ANOVA de medidas repetidas, o η²ₚ da interação fica entre "
    f"{num(rm.eta2_parcial.min(), 2)} e {num(rm.eta2_parcial.max(), 2)}.\n\n"
    f"**O que isso significa.** "
    + ("Os quatro testes concordam em todos os casos: a interação existe também com o mapa como unidade, e o teste "
       "passa a ser defensável contra a crítica de pseudorreplicação. Com o mapa como sujeito, o η²ₚ é grande, porque "
       "o erro passa a ser só a variação da interação entre mapas; ele não se compara ao η²ₚ sobre células do ato 9. "
       "A leitura que se sustenta é a do ato 9: o padrão muda o tamanho das diferenças entre as estratégias, sem "
       "trocar a ordem entre a fórmula e a memória."
       if conc else "Os testes não concordam em todos os casos; ver a tabela.")
    + (" Alguns ajustes do modelo misto avisam que a estimativa está na fronteira do espaço de parâmetros "
       "(uma variância perto de zero); a ANOVA de medidas repetidas, que não depende disso, dá a mesma conclusão."
       if v.avisos.notna().any() else "")))

## Ato 14 — Duas conferências

**Pergunta 1:** por que o ganho estático da Memória-10 aqui é diferente do de agosto?

**Pergunta 2:** os casos em que a memória congelada fica pior que a Manhattan são o colapso de pivô?

In [ ]:
G6 = le2("b6_ganho_estatico_colapso").set_index("estrategia")
P6 = le2("b6_piores_que_manhattan_mapa")
C6 = le2("b6_celulas_cong_manh")
pior = C6[C6.razao_cong_manh > 1]
fora_col = pior[~pior.colapso]
display(Markdown(
    f"**Memória-10 estática.** Agosto publicou {num(G6.loc['Memória-10', 'agosto_publicado'])}× excluindo as células "
    f"(mapa, semente, pivôs) em que a memória colapsou (P8); com elas, o mesmo cálculo de agosto dá "
    f"{num(G6.loc['Memória-10', 'agosto_com_colapso'])}×, que é exatamente o que o v2 mostra "
    f"({num(G6.loc['Memória-10', 'ganho_com_colapso'])}×). O mesmo vale para a Memória-50 "
    f"({num(G6.loc['Memória-50', 'agosto_publicado'])}× contra {num(G6.loc['Memória-50', 'ganho_com_colapso'])}×). "
    f"Não é deduplicação nem população: é a exclusão do colapso. Para o artigo, o número certo é o que usa a mesma "
    f"população do resto da análise: como o v2 mantém o colapso em todo lugar, o estático também tem de mantê-lo "
    f"({num(G6.loc['Memória-10', 'ganho_com_colapso'])}×), com o valor sem colapso como sensibilidade.\n\n"
    f"**Memória congelada pior que a Manhattan.** Por mapa: "
    + "; ".join(f"{r.mapa} ({r.estrategia}, {num(r.ganho)}×)" for r in P6.itertuples())
    + f". Por (mapa, semente, pivôs), {len(pior)} células são piores que a Manhattan; "
    f"{int(pior.colapso.sum())} são exatamente as células de colapso de pivô, e "
    + (", ".join(f"{r.mapa} s{r.semente} com {r.n_pivos} pivôs (razão {num(r.razao_cong_manh, 3)})"
                 for r in fora_col.itertuples()) + " não é colapso, mas é praticamente um empate"
       if len(fora_col) else "nenhuma fica fora delas")
    + ". Sem as células de colapso, nenhum mapa fica pior que a Manhattan."))

## Ato 15 — Um peso basta?

**Pergunta:** a fórmula ganha da Manhattan porque superestima: troca caminhos um pouco mais longos por menos expansões. O Weighted A\* (WA\*, h = w · Manhattan) faz essa mesma troca sem aprender nada. Com caminhos de qualidade igual, a fórmula expande menos que o WA\*?

Calibro o peso w por mapa nos mesmos pares em que a fórmula foi sintetizada (treino, mapa original), até o ρ do WA\* casar com o da fórmula (|Δρ| ≤ 0,005; grade 1,1–3, ampliada até 10 e, no limite, a busca gulosa, GBFS, que é o WA\* com w → ∞). Depois avalio a fórmula e o WA\* com esse peso fixo nos pares fora do treino, no original e nos mapas degradados, como a fórmula, que também não é atualizada. A regra por mapa: **vence** se a fórmula expande menos que o WA\* de ρ logo acima do dela (w_sup), **perde** se expande mais que o de ρ logo abaixo (w_inf), **empata** entre os dois.

In [ ]:
W7 = lambda nome: pd.read_csv(T2 / f"wastar_{nome}.csv")
CAL7, CONT7, EST7, DEC7 = W7("calibracao"), W7("decisao_contagem").set_index("cenario"), W7("estatistica"), W7("decisao")
FRASE7, FORA7, CONF7 = W7("frase").iloc[0], W7("fora_da_curva"), W7("conferencia_w1").iloc[-1]
TRACO7 = pd.read_csv(MEDICOES / "wastar_calibracao_bruta.csv")
E7 = EST7[EST7.recorte == "rho casado (w*)"].set_index("cenario")
cen7 = ["original", "10%", "20%", "30%"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.2), gridspec_kw=dict(width_ratios=[1.1, 1]))
ro = DEC7[(DEC7.cenario == "original") & (DEC7.categoria != "fora_da_curva")].sort_values("razao_wastar_formula")
mm = ro.iloc[len(ro) // 2]
t = TRACO7[(TRACO7.mapa == mm.mapa) & (TRACO7.conjunto == "teste")]
w = t[t.papel != "formula"].drop_duplicates("w").sort_values("w")
f = t[t.papel == "formula"].iloc[0]
a1.plot(w.rho, w.exp_geom, "-o", color=COR["Manhattan"], ms=4, lw=1.4, label="WA* (w crescente)")
g = w[np.isinf(w.w)].iloc[0]; m = w[w.w == 1].iloc[0]
a1.annotate("Manhattan (w = 1)", (m.rho, m.exp_geom), xytext=(6, 0), textcoords="offset points", fontsize=8.5)
a1.scatter(g.rho, g.exp_geom, marker="^", s=55, color=COR["Manhattan"], zorder=3)
a1.annotate("GBFS (w→∞)", (g.rho, g.exp_geom), xytext=(0, 9), textcoords="offset points", fontsize=8.5, ha="center")
a1.scatter(f.rho, f.exp_geom, marker="D", s=60, color=COR["Fórmula"], edgecolor="black", zorder=3, label="Fórmula")
a1.set_yscale("log"); a1.legend(loc="upper right", fontsize=8.5)
a1.set_xlabel("ρ = caminho ÷ ótimo (média geométrica)"); a1.set_ylabel("expansões (média geométrica, log)")
a1.set_title(f"{mm.mapa} (mapa mediano), original", fontsize=10)
for i, c in enumerate(cen7):
    v = DEC7[(DEC7.cenario == c) & (DEC7.categoria != "fora_da_curva")].razao_wastar_formula.sort_values()
    a2.scatter(i + np.linspace(-0.18, 0.18, len(v)), v.values, s=14, color=COR["Fórmula"], alpha=0.45, lw=0)
    a2.plot([i, i], [E7.loc[c, "ic95_bca_inf"], E7.loc[c, "ic95_bca_sup"]], color="black", lw=1.4)
    a2.plot(i, E7.loc[c, "razao_media_geometrica"], "D", color=COR["Fórmula"], mec="black", ms=7)
a2.axhline(1, color="0.5", lw=0.8, ls="--")
a2.set_yscale("log"); a2.set_xticks(range(4), ["original", "10%", "20%", "30%"])
a2.yaxis.set_major_locator(mpl.ticker.LogLocator(subs=(1, 1.5, 2, 3)))
a2.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
a2.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
a2.set_ylabel("expansões do WA* (w*) ÷ fórmula, por mapa")
a2.set_title("razão por nível de degradação", fontsize=10)
a2.text(0, -0.13, "losango: média geométrica nos mapas; barra: IC 95% BCa; pontos: um por mapa",
        transform=a2.transAxes, fontsize=8, color="0.35", va="top")
ro_, rd_ = E7.loc["original", "razao_media_geometrica"], E7.loc["degradado (todos)", "razao_media_geometrica"]
fig.suptitle(("Com o mesmo ρ, o WA* expande " + f"{num(ro_)}× o que a fórmula expande no original e {num(rd_)}× no degradado "
               "(média geométrica)"
              ) if min(ro_, rd_) > 1 else "Com o mesmo ρ, o WA* empata ou ganha da fórmula", x=0.02, ha="left",
             fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "wastar_controle")
plt.show()
cat7 = ["vence", "empata", "perde", "fora da curva"]
conta = lambda c: ", ".join(f"{int(CONT7.loc[c, k])} {k}" for k in cat7 if CONT7.loc[c, k]) + f" (de {int(CONT7.loc[c, 'mapas'])})"
fc = CAL7[CAL7.categoria == "fora_da_curva"]
fo = FORA7[FORA7.cenario == "original"]
ic = lambda c: (f"IC 95% BCa {num(E7.loc[c, 'ic95_bca_inf'])}–{num(E7.loc[c, 'ic95_bca_sup'])}; mediana "
               f"{num(E7.loc[c, 'razao_mediana'])}×, de {num(E7.loc[c, 'razao_min'])}× a {num(E7.loc[c, 'razao_max'])}×")
ab = CONT7.loc["degradado (todos)"]
gn = lambda x: f"{x:g}".replace(".", ",")
display(Markdown(
    f"**Conferência.** Com w = 1, o WA* reproduz a Manhattan já medida em {milhar(CONF7.pares_conferidos)} pares "
    f"(original e os 5 padrões a 30%, semente 42), com {milhar(CONF7.divergentes)} divergências.\n\n"
    f"**Calibração.** w* mediano {gn(CAL7.w_estrela.median())} "
    f"(de {gn(CAL7.w_estrela.min())} a {gn(CAL7.w_estrela.max())}); "
    f"{int(CAL7.grade_ampliada.sum())} mapas precisaram de peso acima de 3 e "
    f"{int((CAL7.categoria == 'casado_gbfs').sum())} só foram cercados com a GBFS. "
    + (f"Em {', '.join(fc.mapa)} nem a GBFS chega ao ρ da fórmula ({', '.join(num(r, 3) for r in fc.rho_gbfs)} contra "
       f"{', '.join(num(r, 3) for r in fc.rho_alvo)}).\n\n" if len(fc) else "\n\n")
    + f"**Regra por mapa** (pares fora do treino; o `arena` não tem nenhum): no original, {conta('original')}; a 10%, "
    f"{conta('10%')}; a 20%, {conta('20%')}; a 30%, {conta('30%')}.\n\n"
    f"**Tamanho do efeito.** Razão WA* ÷ fórmula, média geométrica entre os mapas: {num(ro_)}× no original ({ic('original')}, "
    f"Wilcoxon p = {num(E7.loc['original', 'wilcoxon_p'], 5)}, com Holm sobre os 5 cenários {num(E7.loc['original', 'wilcoxon_p_holm'], 4)}, δ de Cliff {num(E7.loc['original', 'cliff_delta'])}) e "
    f"{num(rd_)}× no degradado ({ic('degradado (todos)')}, p = {num(E7.loc['degradado (todos)', 'wilcoxon_p'], 5)}, Holm {num(E7.loc['degradado (todos)', 'wilcoxon_p_holm'], 4)}); a fórmula expande menos que o WA* "
    f"nos {int(E7.loc['original', 'mapas_formula_expande_menos'])} mapas, em todos os cenários.\n\n"
    f"**O ρ casado escorrega um pouco fora do treino.** A diferença mediana fórmula − WA* é de "
    f"{num(E7.loc['original', 'dif_rho_mediana'], 4)} no original e sobe para {num(E7.loc['30%', 'dif_rho_mediana'], 4)} "
    f"a 30%: no degradado, parte da vantagem vem de caminhos um pouco piores que os do WA*. Mesmo assim, em "
    f"{int(ab.abaixo_da_curva_inteira)} de {int(ab.vence + ab.empata + ab.perde)} mapas a fórmula expande menos que a "
    f"GBFS, o WA* de menos expansões, com ρ menor ou igual ao dela: fica abaixo da curva inteira, qualquer que seja "
    f"o peso.\n\n"
    + (f"**Fora da curva.** No {', '.join(fo.mapa)}, a GBFS expande {', '.join(num(x) for x in fo.razao_gbfs_formula)}× "
       f"o que a fórmula expande e acha caminhos melhores (ρ {', '.join(num(x, 3) for x in fo.rho_gbfs)} contra "
       f"{', '.join(num(x, 3) for x in fo.rho_formula)}): a fórmula faz uma troca entre expansões e qualidade de caminho "
       f"que nenhum peso uniforme sobre a Manhattan reproduz.\n\n" if len(fo) else "")
    + f"**A frase.** {FRASE7.frase}\n\n"
    f"**O que isso significa.** "
    + ("A vantagem da fórmula não é só superestimar: um peso uniforme sobre a Manhattan, ajustado para dar caminhos "
       "tão longos quanto os dela, ainda expande mais. A forma da fórmula guia a busca melhor que a Manhattan escalada, "
       "e isso continua valendo no mapa degradado, onde nenhum dos dois é atualizado."
       if min(ro_, rd_) > 1 else
       "Com o mesmo ρ, um peso uniforme sobre a Manhattan faz o que a fórmula faz: a vantagem dela sobre a Manhattan vem "
       "da superestimação, não da forma.")))

## Ato 16 — A atualização incremental muda o limiar?

**Pergunta:** o custo de atualizar do ato 12 é refazer todas as BFS dos pivôs. Métodos incrementais (Delling e Wagner, LPA\*, D\* Lite) só tocam as distâncias que mudaram. Se poucas mudam, o limiar de buscas a partir do qual atualizar compensa está inflado. Quanto das distâncias muda de fato?

Para cada pivô, comparo a tabela de distâncias antes e depois da degradação, célula a célula. O conjunto que muda, mais as células que a degradação bloqueou, é o mínimo que qualquer método incremental teria de tocar uma vez (U_inc). É um limite inferior generoso: ignora a fila de prioridade, a reabertura de vértices e todo o resto que um método real paga.

In [ ]:
FP8, R8 = le2("b8_fracao_por_padrao"), le2("b8_razao_inc_total")
QS8, FR8, FR8F = le2("b8_q_faixa_resumo"), le2("b8_frase"), le2("b8_frase", T2F)
FA8 = le2("b8_fracao_alterada")
f10 = FA8[(FA8.nivel == 0.1) & (FA8.n_pivos == 100)].iloc[0]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.0), gridspec_kw=dict(width_ratios=[1, 1.1]))
marcas = dict(zip(PADRAO_PT, ["o", "s", "^", "D", "v"]))
for i, (p, pt) in enumerate(PADRAO_PT.items()):
    t = FP8[FP8.padrao == p].sort_values("nivel")
    x = 100 * t.nivel + (i - 2) * 0.5
    a1.fill_between(x, 100 * t.q1, 100 * t.q3, color="0.85", alpha=0.35, lw=0)
    a1.plot(x, 100 * t.mediana, marker=marcas[p], lw=1.6, label=pt)
a1.set_xticks([10, 20, 30]); a1.set_xlabel("nível de degradação (%)")
a1.set_ylabel("distâncias dos pivôs que mudam (%)"); a1.legend(fontsize=8.5, loc="lower right")
a1.set_title(f"Mesmo a 10%, {num(100 * f10.mediana, 0)}% das distâncias mudam", fontsize=10)
q = QS8[(QS8.alternativa == "formula") & (QS8.n_pivos == 100) & (QS8.c.astype(str) == "1")].sort_values("nivel")
lar = 0.26
for k, (col, rot, cor) in enumerate((("q_mud_mediana", "só as distâncias que mudam", "#9ecae1"),
                                     ("q_inc_mediana", "incremental ideal (U_inc)", "#4292c6"),
                                     ("q_total_mediana", "recálculo completo (ato 12)", "#08306b"))):
    a2.bar(np.arange(3) + (k - 1) * lar, q[col], lar, color=cor, label=rot)
a2.set_xticks(range(3)); a2.set_xticklabels([f"{int(100 * v)}%" for v in q.nivel])
a2.set_ylabel("Q* (buscas entre mudanças)"); a2.legend(fontsize=8.5, loc="upper left")
a2.set_title("Memória-100 contra a fórmula, c = 1: as três contas dão o mesmo limiar", fontsize=10)
fig.suptitle("Atualizar de forma incremental não encurta o limiar: a degradação muda quase todas as distâncias",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "incremental_distancias")
plt.show()

In [ ]:
r100 = R8[R8.n_pivos == 100].set_index("nivel")
f1, f1f = FR8[FR8.c.astype(str) == "1"].iloc[0], FR8F[FR8F.c.astype(str) == "1"].iloc[0]
pct8 = lambda x: num(100 * x, 1) + "%"
display(Markdown(
    f"Com a Memória-100, a fração de distâncias que muda é {pct8(FA8[(FA8.n_pivos == 100)].set_index('nivel').mediana[0.1])}, "
    f"{pct8(FA8[(FA8.n_pivos == 100)].set_index('nivel').mediana[0.2])} e "
    f"{pct8(FA8[(FA8.n_pivos == 100)].set_index('nivel').mediana[0.3])} a 10, 20 e 30% (mediana entre os 17 mapas). "
    f"U_inc ÷ U_total vale {pct8(r100.mediana[0.1])}, {pct8(r100.mediana[0.2])} e {pct8(r100.mediana[0.3])}: a 20% e 30% o "
    f"limite inferior do incremental passa do custo do recálculo completo, porque as células bloqueadas se somam ao que "
    f"mudou e uma BFS nova só visita o que continua livre.\n\n"
    f"Contra a fórmula (Memória-100, c = 1, todos os níveis), o limiar fica em {milhar(f1.q_inc_mediana)} buscas com o "
    f"incremental ideal, {milhar(f1.q_mud_mediana)} contando só as distâncias que mudam e {milhar(f1.q_total_mediana)} com "
    f"o recálculo completo (mediana entre os mapas). Fora do treino: {milhar(f1f.q_inc_mediana)}, "
    f"{milhar(f1f.q_mud_mediana)} e {milhar(f1f.q_total_mediana)}.\n\n"
    f"**O que isso significa.** A crítica de que o Q\\* do ato 12 está inflado por usar o pior caso não se sustenta "
    f"neste problema. Quando um corredor fecha, o desvio muda a distância de tudo o que fica depois dele na árvore de "
    f"caminhos do pivô, então quase toda a tabela precisa ser reescrita. O limiar vira uma faixa estreita, e o mote "
    f"continua o mesmo: abaixo de algumas centenas de buscas entre mudanças, a fórmula é a melhor escolha."))

## Ato 17 — A fórmula ganhou por sorte do sorteio?

**Pergunta:** cada mapa tinha uma fórmula só, de uma síntese que não podia ser repetida (o algoritmo genético sorteava da entropia do sistema). A vitória da fórmula sobre a memória congelada pode ter sido sorte. Se eu sintetizar de novo, com semente fixa, a conclusão continua?

Ressintetizei 3 fórmulas por mapa (sementes 1, 2 e 3), com tudo o mais igual a março: população 80, 100 gerações, os mesmos pares de treino, as mesmas flags de compilação. Rodar duas vezes com a mesma semente dá a mesma fórmula, com 1 ou 6 threads. Cada fórmula foi sintetizada no mapa original e reutilizada, sem nova síntese, nos 5 padrões × 3 níveis × 5 sementes, como a original.

In [ ]:
AV11 = le2("ressintese_avaliacao")
CT11 = le2("ressintese_contagem_vitorias").set_index("formulas_que_vencem")
VR11, TIP11, FR11 = le2("ressintese_variancia"), le2("ressintese_original_tipica"), le2("ressintese_frase").iloc[0]
GB11 = le2("ressintese_gbfs_contagem")
FORM11 = pd.read_csv(ANALISE / "formulas_v2_seed" / "formulas_v2_seed.csv")
D11 = AV11[AV11.cenario == "degradado (todos)"]
ordem = D11[D11.run_seed == 0].sort_values("ganho").mapa.tolist()
fig, ax = plt.subplots(figsize=(11, 4.2))
xs = {m: i for i, m in enumerate(ordem)}
for m in ordem:
    v = D11[D11.mapa == m]
    ax.hlines(v.ganho_mem100_congelada.iloc[0], xs[m] - 0.42, xs[m] + 0.42, color=COR["Memória-100"], lw=2, ls="--")
for rs, mk, rot in ((1, "o", "semente 1"), (2, "s", "semente 2"), (3, "^", "semente 3"), (0, "D", "original (março)")):
    q = D11[D11.run_seed == rs]
    ax.scatter([xs[m] + (rs - 1.5 if rs else 0) * 0.13 for m in q.mapa], q.ganho, marker=mk, s=42 if rs == 0 else 30,
               color=COR["Fórmula"] if rs == 0 else "white", edgecolor=COR["Fórmula"], lw=1.2, zorder=3, label=rot)
ax.plot([], [], color=COR["Memória-100"], lw=2, ls="--", label="Memória-100 sem atualização")
ax.set_yscale("log"); ax.set_xticks(range(len(ordem))); ax.set_xticklabels(ordem, rotation=40, ha="right")
ax.set_ylabel("ganho sobre a Manhattan no degradado (log)")
ax.legend(ncol=5, fontsize=8.5, loc="upper left")
y4 = int(FR11.mapas_4_de_4); nm = int(FR11.n_mapas)
ax.set_title(f"As 4 sínteses vencem a memória congelada em {y4} de {nm} mapas, e variam menos entre si que os mapas",
             fontsize=11)
fig.tight_layout()
salvar(fig, "ressintese_ganho")
plt.show()

In [ ]:
cont = CT11["degradado (todos)"]
vg = VR11.set_index("medida")
porf = D11.groupby("formula").vence_mem100_congelada.sum().astype(int)
gb = GB11[GB11.cenario == "degradado (todos)"].set_index("run_seed")
tip = TIP11.set_index("cenario").loc["degradado (todos)"]
display(Markdown(
    f"Mapas por número de fórmulas (de 4) que vencem a Memória-100 congelada no degradado: "
    + ", ".join(f"{k} de 4 em {int(v)}" for k, v in cont.sort_index(ascending=False).items() if v) + ". "
    f"Por fórmula: " + ", ".join(f"{f} vence em {v}" for f, v in porf.items()) + f" de {nm} mapas.\n\n"
    f"Variação entre as sínteses de um mesmo mapa: CV do ganho {num(vg.loc['ganho', 'cv_entre_sinteses_mediana'], 3)} "
    f"(mediana entre os mapas; de {num(vg.loc['ganho', 'cv_entre_sinteses_min'], 3)} a "
    f"{num(vg.loc['ganho', 'cv_entre_sinteses_max'], 3)}), contra {num(vg.loc['ganho', 'cv_entre_mapas'], 3)} entre os mapas; "
    f"CV do ρ {num(vg.loc['rho', 'cv_entre_sinteses_mediana'], 4)} contra {num(vg.loc['rho', 'cv_entre_mapas'], 4)}.\n\n"
    f"Contra a GBFS, nos pares fora do treino, a fórmula expande menos em "
    + ", ".join(f"{int(r.mapas_formula_expande_menos)} de {int(r.n_mapas)} mapas ({'original' if s == 0 else f'semente {s}'})"
                for s, r in gb.iterrows()) + ". "
    f"A fórmula de março fica entre a pior e a melhor das 3 novas em {int(tip.mapas_original_entre_pior_e_melhor)} mapas, "
    f"acima de todas em {int(tip.mapas_original_acima_de_todas)} e abaixo de todas em "
    f"{int(tip.mapas_original_abaixo_de_todas)}.\n\n"
    f"> {FR11.frase}\n\n"
    f"**O que isso significa.** "
    + ("A vantagem não é sorte do sorteio: as sínteses independentes reencontram fórmulas diferentes no texto, mas com "
       "desempenho parecido, e todas ficam do mesmo lado da memória congelada. A variação entre sínteses é pequena perto da "
       "variação entre mapas, então o que decide o ganho é o mapa, não a semente."
       if y4 >= nm / 2 and vg.loc['ganho', 'cv_entre_sinteses_mediana'] < vg.loc['ganho', 'cv_entre_mapas'] else
       "A conclusão depende da síntese em parte dos mapas; ver a contagem acima.")))

## Ato 18 — Quanto custa uma expansão de verdade?

**Pergunta:** o ponto de equilíbrio (ato 12) supõe que uma expansão do A\* custa o mesmo que *c* visitas da BFS, com *c* entre 1 e 10, e que toda expansão custa o mesmo, qualquer que seja a heurística. Medido no relógio, quanto vale *c*? E a ordem entre as estratégias continua a mesma quando a métrica é tempo?

Medição com uma thread, preso a um núcleo, com a máquina parada: 200 pares fora do treino por mapa (estratificados pelo comprimento do caminho), no original e no Orgânico a 30%, 1 execução de aquecimento e a mediana de 5. O tempo por expansão tem duas leituras. A razão tempo ÷ expansões inclui o custo fixo de cada busca, que neste código é alocar três matrizes do tamanho do mapa. A inclinação do tempo contra as expansões (Theil-Sen) é o custo de uma expansão a mais, e é ela que entra no ponto de equilíbrio, porque o custo fixo é o mesmo para todas as heurísticas.

In [ ]:
C10, CR10 = le2("c_medido"), le2("c_medido_resumo").set_index("n_pivos")
G10, QS10, FR10 = le2("tempo_ganho_sobre_manhattan"), le2("tempo_q_estrela_resumo"), le2("tempo_frase").iloc[0]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.2))
x = CR10.index.values
for col, rot, ls, mk in (("c", "custo marginal (inclinação)", "-", "o"), ("c_razao", "tempo ÷ expansões", "--", "s")):
    a1.plot(x, CR10[f"{col}_median"], ls=ls, marker=mk, color=COR["Memória-100"], lw=1.8, label=rot)
    a1.fill_between(x, CR10[f"{col}_q1"], CR10[f"{col}_q3"], color=COR["Memória-50"], alpha=0.15, lw=0)
a1.axhspan(1, 10, color="0.9", zorder=0); a1.text(11, 1.2, "faixa assumida (1 a 10)", fontsize=8.5, color="0.4")
a1.set_xscale("log"); a1.set_yscale("log"); a1.set_xticks(x); a1.set_xticklabels(x)
a1.set_xlabel("número de pivôs"); a1.set_ylabel("c (visitas da BFS por expansão)"); a1.legend(fontsize=8.5)
a1.set_title(f"c vai de {num(FR10.c_min, 1)} a {num(FR10.c_max, 1)} e cresce com os pivôs", fontsize=10)
d = G10[G10.estado == "degradado"]
for r in d.itertuples():
    e = r.estrategia.split(" ")[0]
    cor = COR.get(e, "0.3")
    a2.scatter(r.ganho_expansoes, r.ganho_tempo, color=cor if "sem" not in r.estrategia else "white",
               edgecolor=cor, s=40, marker=MARCA.get(e, "^"), zorder=3)
    a2.annotate(r.estrategia.replace("Memória", "Mem").replace(" com atualização", " (atual.)")
                .replace(" sem atualização", " (cong.)"), (r.ganho_expansoes, r.ganho_tempo), fontsize=7.5,
                xytext=(4, 2), textcoords="offset points")
lim = [0.15, max(d.ganho_expansoes.max(), d.ganho_tempo.max()) * 1.4]
a2.plot(lim, lim, color="0.7", lw=0.8, ls=":"); a2.axhline(1, color="0.7", lw=0.8)
a2.set_xscale("log"); a2.set_yscale("log")
a2.set_xlabel("ganho sobre a Manhattan em expansões"); a2.set_ylabel("ganho sobre a Manhattan em tempo")
a2.set_title("Orgânico 30%: em tempo, a memória perde o que ganha em expansões", fontsize=10)
fig.suptitle("Uma expansão da memória custa várias visitas da BFS, e o custo cresce com o número de pivôs",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "c_medido")
plt.show()

In [ ]:
qf = QS10[QS10.alternativa == "formula"].set_index("n_pivos")
ordem_t = d.sort_values("ganho_tempo", ascending=False).estrategia.tolist()
ordem_e = d.sort_values("ganho_expansoes", ascending=False).estrategia.tolist()
display(Markdown(
    f"Ambiente: `{FR10.ambiente}`.\n\n"
    f"c marginal (mediana entre os mapas): " + ", ".join(f"{num(CR10.c_median[n], 1)} com {n} pivôs" for n in x)
    + "; pela razão tempo ÷ expansões, " + ", ".join(f"{num(CR10.c_razao_median[n], 1)}" for n in x) + ".\n\n"
    f"Q\\* contra a fórmula com o c medido (mediana entre os mapas, todos os níveis): "
    + ", ".join(f"{milhar(qf.q_inc[n])} a {milhar(qf.q_total[n])} com {n} pivôs" for n in x)
    + ". Cobrando de cada heurística o seu próprio custo por expansão: "
    + ", ".join(("nunca compensa" if not np.isfinite(qf.q_tempo_total[n]) else milhar(qf.q_tempo_total[n]))
                + f" com {n}" for n in x) + " pivôs.\n\n"
    f"Ordem pelo ganho em tempo no Orgânico 30%: {', '.join(ordem_t)}. Em expansões: {', '.join(ordem_e)}.\n\n"
    f"> {FR10.frase}\n\n"
    f"**O que isso significa.** O c assumido no ato 12 subestima o custo de uma expansão da memória: ela lê k pivôs por "
    f"nó, e o c cresce com k. Com o c medido, o limiar do ato 12 cai, e a vantagem da memória em expansões não se "
    f"converte inteira em tempo. Isso favorece a fórmula, não a memória: a conclusão do artigo fica mais forte em tempo do "
    f"que em expansões. O número em tempo depende desta implementação (alocação por busca, std::function por chamada); "
    f"o que se generaliza é a direção: a memória paga por nó um custo que cresce com o número de pivôs."))

## Ato 19 — Conferências para o revisor

**Pergunta:** as afirmações de apoio que o artigo usa batem com os dados?

In [ ]:
R9 = le2("b9_resumo")
display(Markdown("\n".join(f"- **{r.item}.** {r.resposta}" for r in R9.itertuples())))
EF9 = le2("efeito_pareado")
P9 = le2("b9_c6_permutacao_10000")
display(Markdown(
    f"**O que isso significa.** O Weighted A\\* respeita a garantia ρ ≤ w em todos os pares; nenhum pivô bloqueado gerou "
    f"caminho maior que o ótimo; e o teste de permutação com {milhar(P9.n_permutacoes.iloc[0])} permutações dá o menor p "
    f"possível em todos os {len(P9)} casos, como com 200. A correlação rank-biserial pareada acompanha o sinal do δ de "
    f"Cliff em {int(EF9.mesmo_sinal.sum())} de {len(EF9)} comparações e é maior em módulo, como se espera de um tamanho "
    f"de efeito que usa o pareamento por mapa."))

## Ato 20 — Fechamento

**Pergunta:** o que mudou do WPerformance (v1) para cá?

In [ ]:
gd = G[(G.nivel == "todos")]
def melhor_glob(metrica, regime, mostra=1):
    q = gd[(gd.metrica == metrica) & (gd.regime == regime)].set_index("estrategia").media_geometrica.sort_values()
    if mostra == 1:
        return q.index[0]
    return q.index[0] + " (" + "; ".join(f"{e} {num(v, 3)}" for e, v in q.iloc[:mostra].items()) + ")"
q1 = np.exp(DELTA_V1.apply(np.log).mean()).sort_values()
v1_menos_degradada = q1.index[0] + " (" + "; ".join(f"{e} {num(v, 3)}" for e, v in q1.iloc[:3].items()) + ")"
v1_inv = POSTOS_V1.posto_estatico.rank().corr(POSTOS_V1.posto_degradado.rank())
sob30 = SOB[SOB.nivel == 0.3]
tab = pd.DataFrame([
    ("Os pares medidos são os mesmos em todos os padrões e níveis",
     "não" if sob30.taxa_v1.nunique() > 1 else "sim", "sim" if (SOB.taxa_v2 == 1).all() else "não"),
    ("A heurística menos degradada (δ, memória com atualização)", v1_menos_degradada,
     melhor_glob("delta", "com_atualizacao", mostra=3)),
    ("A melhor heurística no mapa degradado, em expansões, com atualização", "não medido",
     melhor_glob("absoluta", "com_atualizacao")),
    ("A melhor heurística no mapa degradado, em expansões, sem atualização", "não medido",
     melhor_glob("absoluta", "sem_atualizacao")),
    ("A ordem estática se inverte sob degradação (δ)",
     "por completo" if np.isclose(v1_inv, -1) else ("em parte" if v1_inv < 0 else "não"),
     "por completo, sem atualização" if inverte else "em parte"),
    ("Interação estratégia × padrão (η²ₚ, OLS sobre células, δ com atualização)",
     num(e1["estratégia × padrão"], 3), num(e2["estratégia × padrão"], 3)),
    ("A interação resiste com o mapa como unidade (medidas repetidas)", "não medido",
     "sim" if (le2("b5_modelo_misto").query("modelo == 'anova_rm_mapa'").p < 0.05).all() else "não"),
    ("Caminho da fórmula: excesso médio sobre o ótimo (estático → degradado)", "não medido",
     f"{num(100 * (rs.rho_media_geometrica - 1), 1)}% → {num(100 * (rd.rho_media_geometrica - 1), 1)}%"),
    ("A memória continua achando o caminho ótimo (Proposição A)", "não medido",
     "sim" if viol == 0 else f"não ({milhar(viol)} violações)"),
    ("Expansões do WA* ÷ fórmula com o mesmo ρ (média geométrica, original → degradado)", "não medido",
     f"{num(E7.loc['original', 'razao_media_geometrica'])}× → {num(E7.loc['degradado (todos)', 'razao_media_geometrica'])}×"),
], columns=["conclusão", "v1 (WPerformance)", "v2"])
display(tab.style.hide(axis="index"))

In [ ]:
import re
logs = sorted((V2 / "logs").glob("*.log"))
p11 = {}
for f in logs:
    m = re.search(r"(\d+) divergencias declaradas", f.read_text())
    if m:
        p11[f.stem.split("_")[0]] = int(m.group(1))
nperm = int(INTER.n_permutacoes.iloc[0])
display(Markdown(
    f"**Limitações que continuam valendo**\n\n"
    f"- **Tempo (P3):** os tempos da execução principal não sustentam conclusão; o tempo foi medido à parte, com "
    f"repetição e aquecimento, no ato 18, para o c do ponto de equilíbrio.\n"
    f"- **Uma fórmula por mapa (P5):** a síntese de março não é reprodutível; o ato 17 ressintetiza 3 fórmulas por "
    f"mapa com semente fixa e mostra que a conclusão não depende do sorteio.\n"
    f"- **Colapso de pivô (P8):** mantido sem correção, para comparar com o código de março; entra nas médias "
    f"({len(COLAPSO)} combinações de mapa e semente, ato 7).\n"
    f"- **Linha de base (P11):** "
    + (", ".join(f"no `{m}`, {n} problemas da fórmula divergem de março por arredondamento" for m, n in p11.items())
       if p11 else "nenhuma divergência declarada nos logs")
    + " (detalhes em `docs/06-protocolo-v2.md`).\n"
    f"- **Deformação dos padrões:** um padrão do v2 não é o mesmo obstáculo do v1 (ato 4); comparar padrões entre "
    f"protocolos é só qualitativo.\n"
    f"- **Permutação:** com {nperm} permutações, o menor p possível é 1/{nperm + 1}; os p de permutação mostram só "
    f"que nenhuma permutação chegou perto do observado."))

In [ ]:
display(Markdown(f"_Notebook executado em {time.time() - INICIO:.0f} s._"))